In [ ]:
# In[1]:

import pandas as pd
import pytz

# reuse timezone object per RULE 10
tz = pytz.timezone('Asia/Shanghai')

# Load CSVs (they are in working directory)
df_metric = pd.read_csv('metric.csv')
df_trace = pd.read_csv('trace.csv')
df_log = pd.read_csv('log.csv')
df_error = pd.read_csv('error_logs.csv')

# Helper to compute compact timestamp summary (epoch and localized string)
def ts_summary(s):
    if s.empty:
        return {'min_epoch': None, 'max_epoch': None, 'min_local': None, 'max_local': None}
    min_epoch = int(s.min())
    max_epoch = int(s.max())
    min_local = pd.to_datetime(min_epoch, unit='s', utc=True).tz_convert(tz)
    max_local = pd.to_datetime(max_epoch, unit='s', utc=True).tz_convert(tz)
    # keep compact string form
    return {
        'min_epoch': min_epoch,
        'max_epoch': max_epoch,
        'min_local': min_local.strftime('%Y-%m-%d %H:%M:%S %Z'),
        'max_local': max_local.strftime('%Y-%m-%d %H:%M:%S %Z')
    }

# 1) metric.csv summary
metric_ts = ts_summary(df_metric['timestamp'])
metric_total = len(df_metric)
metric_top_pairs = (
    df_metric.groupby(['cmdb_id', 'kpi_name'])
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
    .head(20)
    .reset_index(drop=True)
)

metric_summary = pd.DataFrame([{
    'total_rows': metric_total,
    'min_timestamp_epoch': metric_ts['min_epoch'],
    'max_timestamp_epoch': metric_ts['max_epoch'],
    'min_timestamp_local': metric_ts['min_local'],
    'max_timestamp_local': metric_ts['max_local']
}])

# 2) trace.csv summary
trace_ts = ts_summary(df_trace['timestamp'])
trace_total = len(df_trace)
trace_top_pairs = (
    df_trace.groupby(['cmdb_id', 'trace_name'])
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
    .head(20)
    .reset_index(drop=True)
)

trace_summary = pd.DataFrame([{
    'total_rows': trace_total,
    'min_timestamp_epoch': trace_ts['min_epoch'],
    'max_timestamp_epoch': trace_ts['max_epoch'],
    'min_timestamp_local': trace_ts['min_local'],
    'max_timestamp_local': trace_ts['max_local']
}])

# 3) log.csv summary
log_ts = ts_summary(df_log['timestamp'])
log_total = len(df_log)
log_top_pairs = (
    df_log.groupby(['cmdb_id', 'log_name'])
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
    .head(20)
    .reset_index(drop=True)
)

log_summary = pd.DataFrame([{
    'total_rows': log_total,
    'min_timestamp_epoch': log_ts['min_epoch'],
    'max_timestamp_epoch': log_ts['max_epoch'],
    'min_timestamp_local': log_ts['min_local'],
    'max_timestamp_local': log_ts['max_local']
}])

# 4) error_logs.csv summary
error_ts = ts_summary(df_error['timestamp'])
error_total = len(df_error)
error_cmdb_counts = (
    df_error['cmdb_id']
    .value_counts()
    .reset_index()
    .rename(columns={'index':'cmdb_id', 'cmdb_id':'count'})
    .head(20)
)

# sample up to 5 rows with formatted timestamp (keep compact)
if not df_error.empty:
    df_error_samples = df_error.sort_values('timestamp').head(5).copy()
    df_error_samples['timestamp_local'] = pd.to_datetime(df_error_samples['timestamp'], unit='s', utc=True).dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S %Z')
    error_samples = df_error_samples[['timestamp_local','cmdb_id','message']].reset_index(drop=True)
else:
    error_samples = pd.DataFrame(columns=['timestamp_local','cmdb_id','message'])

error_summary = pd.DataFrame([{
    'total_rows': error_total,
    'min_timestamp_epoch': error_ts['min_epoch'],
    'max_timestamp_epoch': error_ts['max_epoch'],
    'min_timestamp_local': error_ts['min_local'],
    'max_timestamp_local': error_ts['max_local']
}])

# Final compact outputs (ordered per request). Use variable names so IPython displays them.
metric_summary, metric_top_pairs, trace_summary, trace_top_pairs, log_summary, log_top_pairs, error_summary, error_cmdb_counts, error_samples

```
Out[1]:
```
```python
# Concise plain-English summary of the previously displayed results
summary = (
    "Summary of telemetry CSVs:\n\n"
    "metric.csv:\n"
    "- Total rows: 321,992\n"
    "- Time range (epoch): 1647810000 to 1647813600\n"
    "- Time range (Asia/Shanghai): 2022-03-21 05:00:00 CST to 2022-03-21 06:00:00 CST\n"
    "- Top (cmdb_id, kpi_name) pairs (most frequent, sample):\n"
    "  * shippingservice2-0 (mesh.source...): 61 occurrences\n"
    "  * adservice (app.grpc.count, app.grpc.mrt, app.grpc.rr, app.grpc.sr, app.http.count, ...): 61 occurrences each\n"
    "  Note: many adservice runtime/JVM metrics also appear frequently (counts ~61).\n\n"
    "trace.csv:\n"
    "- Total rows: 28,272\n"
    "- Time range (epoch): 1647811800 to 1647813540\n"
    "- Time range (Asia/Shanghai): 2022-03-21 05:30:00 CST to 2022-03-21 05:59:00 CST\n"
    "- Top (cmdb_id, trace_name) pairs (sample):\n"
    "  * shippingservice-2, trace.from_frontend-1.row_count: 30 occurrences\n"
    "  * adservice-0, trace.from_frontend-0.duration_mean / p95 / error_rate: 30 occurrences each\n"
    "  * shippingservice-0 and cartservice-0 also show many frequent trace metrics (counts ~30).\n\n"
    "log.csv:\n"
    "- Total rows: 1,798\n"
    "- Time range (epoch): 1647811800 to 1647813600\n"
    "- Time range (Asia/Shanghai): 2022-03-21 05:30:00 CST to 2022-03-21 06:00:00 CST\n"
    "- Top (cmdb_id, log_name) pairs (sample):\n"
    "  * cartservice-0/1/2, log.row_count and log.error_count: 31 occurrences each\n"
    "  * currencyservice-*, frontend-0, productcatalogservice-2, recommendationservice-* also frequent (counts ~31).\n\n"
    "error_logs.csv:\n"
    "- Total rows: 0 (no error log records)\n"
    "- No cmdb_id entries and no sample rows to show.\n\n"
    "Overall: metric data is very dense (322k rows) across 05:00–06:00 CST, trace and log data concentrate in 05:30–06:00 CST, and there are no error_log entries in the dataset."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(   total_rows  min_timestamp_epoch  max_timestamp_epoch      min_timestamp_local      max_timestamp_local
0      321992           1647810000           1647813600  2022-03-21 05:00:00 CST  2022-03-21 06:00:00 CST,                cmdb_id                                           kpi_name  count
0   shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....     61
1            adservice                                     app.grpc.count     61
2            adservice                                       app.grpc.mrt     61
3            adservice                                        app.grpc.rr     61
4            adservice                                        app.grpc.sr     61
5            adservice                                     app.http.count     61
6            adservice                                       app.http.mrt     61
7            adservice                                        app.http.rr     61
8            adservice                                        app.http.sr     61
9            adservice    runtime.java_lang_ClassLoading_LoadedClassCount     61
10           adservice  runtime.java_lang_ClassLoading_TotalLoadedClas...     61
11           adservice  runtime.java_lang_ClassLoading_UnloadedClassCount     61
12           adservice             runtime.java_lang_ClassLoading_Verbose     61
13           adservice  runtime.java_lang_Compilation_CompilationTimeM...     61
14           adservice  runtime.java_lang_Compilation_TotalCompilation...     61
15           adservice  runtime.java_lang_GarbageCollector_CollectionC...     61
16           adservice  runtime.java_lang_GarbageCollector_CollectionC...     61
17           adservice  runtime.java_lang_GarbageCollector_CollectionT...     61
18  shippingservice2-0  container.node-6.container_network_transmit_MB...     61
19  shippingservice2-0  container.node-6.container_network_transmit_er...     61,    total_rows  min_timestamp_epoch  max_timestamp_epoch      min_timestamp_local      max_timestamp_local
0       28272           1647811800           1647813540  2022-03-21 05:30:00 CST  2022-03-21 05:59:00 CST,               cmdb_id                           trace_name  count
0   shippingservice-2      trace.from_frontend-1.row_count     30
1         adservice-0  trace.from_frontend-0.duration_mean     30
2         adservice-0   trace.from_frontend-0.duration_p95     30
3         adservice-0     trace.from_frontend-0.error_rate     30
4   shippingservice-0  trace.from_frontend-0.duration_mean     30
5   shippingservice-0   trace.from_frontend-0.duration_p95     30
6   shippingservice-0     trace.from_frontend-0.error_rate     30
7   shippingservice-0      trace.from_frontend-0.row_count     30
8   shippingservice-0  trace.from_frontend-1.duration_mean     30
9   shippingservice-0   trace.from_frontend-1.duration_p95     30
10  shippingservice-0     trace.from_frontend-1.error_rate     30
11  shippingservice-0      trace.from_frontend-1.row_count     30
12      cartservice-0  trace.from_frontend-2.duration_mean     30
13      cartservice-0   trace.from_frontend-2.duration_p95     30
14      cartservice-0     trace.from_frontend-2.error_rate     30
15      cartservice-0      trace.from_frontend-2.row_count     30
16      cartservice-0             trace.self.duration_mean     30
17      cartservice-0              trace.self.duration_p95     30
18      cartservice-0                trace.self.error_rate     30
19      cartservice-0                 trace.self.row_count     30,    total_rows  min_timestamp_epoch  max_timestamp_epoch      min_timestamp_local      max_timestamp_local
0        1798           1647811800           1647813600  2022-03-21 05:30:00 CST  2022-03-21 06:00:00 CST,                     cmdb_id         log_name  count
0             cartservice-2    log.row_count     31
1             cartservice-2  log.error_count     31
2             cartservice-1    log.row_count     31
3             cartservice-1  log.error_count     31
4             cartservice-0    log.row_count     31
5             cartservice-0  log.error_count     31
6         currencyservice-0  log.error_count     31
7         currencyservice-0    log.row_count     31
8         currencyservice-2  log.error_count     31
9         currencyservice-2    log.row_count     31
10        currencyservice-1  log.error_count     31
11        currencyservice-1    log.row_count     31
12               frontend-0    log.row_count     31
13               frontend-0  log.error_count     31
14  productcatalogservice-2  log.error_count     31
15  productcatalogservice-2    log.row_count     31
16  recommendationservice-0  log.error_count     31
17  recommendationservice-0    log.row_count     31
18  recommendationservice-2    log.row_count     31
19  recommendationservice-2  log.error_count     31,    total_rows min_timestamp_epoch max_timestamp_epoch min_timestamp_local max_timestamp_local
0           0                None                None                None                None, Empty DataFrame
Columns: [count, count]
Index: [], Empty DataFrame
Columns: [timestamp_local, cmdb_id, message]
Index: [])```
```

In [ ]:
# In[2]:

# Compute global thresholds (P95 and P5) per (cmdb_id, kpi_name) using entire metric series (before windowing)
# Reuse df_metric loaded earlier in the kernel.

start_window = 1647811800
end_window = 1647813600

# 1) global thresholds
p95 = df_metric.groupby(['cmdb_id', 'kpi_name'])['value'].quantile(0.95).rename('global_P95')
p5 = df_metric.groupby(['cmdb_id', 'kpi_name'])['value'].quantile(0.05).rename('global_P5')
thresholds = pd.concat([p95, p5], axis=1).reset_index()

# 2) filter to incident window
window = df_metric[(df_metric['timestamp'] >= start_window) & (df_metric['timestamp'] <= end_window)].copy()

# 3) compute window-based metrics
# merge thresholds into window rows so we can compare per-row against global thresholds
window = window.merge(thresholds, on=['cmdb_id', 'kpi_name'], how='left')

# create indicator columns for anomalies within window
window['is_high'] = (window['value'] > window['global_P95']).astype(int)
window['is_low'] = (window['value'] < window['global_P5']).astype(int)

# aggregated stats from window rows
grp = window.groupby(['cmdb_id', 'kpi_name'])

agg_stats = grp.agg(
    total_points_in_window = ('value', 'size'),
    anomalies_high = ('is_high', 'sum'),
    anomalies_low = ('is_low', 'sum'),
    max_value_in_window = ('value', 'max'),
    min_value_in_window = ('value', 'min'),
).reset_index()

# earliest timestamps for high/low anomalies (if any)
earliest_high = window.loc[window['is_high'] == 1].groupby(['cmdb_id', 'kpi_name'])['timestamp'].min().rename('earliest_high_ts')
earliest_low = window.loc[window['is_low'] == 1].groupby(['cmdb_id', 'kpi_name'])['timestamp'].min().rename('earliest_low_ts')

# combine aggregated stats and earliest timestamps
agg_stats = agg_stats.merge(earliest_high.reset_index(), on=['cmdb_id','kpi_name'], how='left')
agg_stats = agg_stats.merge(earliest_low.reset_index(), on=['cmdb_id','kpi_name'], how='left')

#  Include pairs that exist in thresholds but had zero points in window:
# left-join thresholds with agg_stats so all pairs are present
final = thresholds.merge(agg_stats, on=['cmdb_id','kpi_name'], how='left')

# Fill NaNs for counts and min/max where there were no window points
final['total_points_in_window'] = final['total_points_in_window'].fillna(0).astype(int)
final['anomalies_high'] = final['anomalies_high'].fillna(0).astype(int)
final['anomalies_low'] = final['anomalies_low'].fillna(0).astype(int)
final['max_value_in_window'] = final['max_value_in_window']  # keep NaN if no data
final['min_value_in_window'] = final['min_value_in_window']
# earliest timestamps: keep NaN for None; will format later
final['earliest_high_ts'] = final['earliest_high_ts']
final['earliest_low_ts'] = final['earliest_low_ts']

# compute anomalies total for sorting
final['anomalies_total'] = final['anomalies_high'] + final['anomalies_low']

# 4) Prepare output:
# total number of distinct pairs processed
distinct_count = final.shape[0]

# Decide sorting: if no anomalies anywhere, sort by total_points_in_window desc; else by anomalies_total desc
if final['anomalies_total'].sum() == 0:
    sort_cols = ['total_points_in_window', 'cmdb_id', 'kpi_name']
    ascending = [False, True, True]
else:
    sort_cols = ['anomalies_total', 'total_points_in_window', 'cmdb_id', 'kpi_name']
    ascending = [False, False, True, True]

top20 = final.sort_values(sort_cols, ascending=ascending).head(20).copy()

# Format earliest timestamps into Asia/Shanghai localized strings; keep None where missing
import pytz
tz = pytz.timezone('Asia/Shanghai')

def fmt_ts(x):
    if pd.isna(x):
        return None
    return pd.to_datetime(int(x), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')

top20['earliest_high_ts'] = top20['earliest_high_ts'].apply(fmt_ts)
top20['earliest_low_ts'] = top20['earliest_low_ts'].apply(fmt_ts)

# Select and order the requested columns for compact display
top20_summary = top20[[
    'cmdb_id', 'kpi_name',
    'total_points_in_window',
    'anomalies_high', 'earliest_high_ts',
    'anomalies_low', 'earliest_low_ts',
    'max_value_in_window', 'min_value_in_window',
    'global_P95', 'global_P5'
]]

# Display compact results: distinct_count and top20_summary
distinct_count, top20_summary

```
Out[2]:
```
- Total distinct (cmdb_id, kpi_name) pairs processed: 5,298.

- Anomalies exist (we sorted by anomalies_high+anomalies_low). Top-20 highlights (compact summary):
  - All top-20 pairs have 31 points in the incident window (timestamps between 2022-03-21 05:30–06:00 CST).
  - Many top entries show anomalies_high = 3 and anomalies_low = 3 (anomalies_total = 6) for each pair — e.g., metrics from:
    - adservice-2 (container_cpu_cfs_periods)
    - adservice2 (several JVM/OS metrics)
    - adservice2-0 (container CPU seconds / mesh metrics)
    - cartservice2-0 (container_network_receive MB…)
    - checkoutservice-1 (multiple mesh source/destination metrics)
  - Earliest high anomaly timestamps in the top results range around 2022-03-21 05:33:00–05:48:00 CST depending on metric; earliest low anomalies similarly occur around 05:30:00–05:51:00 CST.
  - Example numeric ranges (per metric): max/min values in-window and global thresholds vary by metric (samples from the top rows):
    - container_cpu_cfs_periods (adservice-2): max ≈ 125.17, min ≈ 61.50, global_P95 ≈ 116.83, global_P5 ≈ 69.50
    - runtime memory / CPU metrics (adservice2): values in millions or small fractions depending on metric; global P95/P5 provided per-row in the table.

- Interpretation: multiple metrics across several services/pods experienced both high-side and low-side excursions relative to their global P95/P5 thresholds during the incident window. The top anomalies are small in absolute count (3 high + 3 low among 31 samples) but appear across many related metrics for the same components (notably adservice, adservice2, checkoutservice-1, etc.), suggesting correlated deviations in those services during ~05:30–05:50 CST.

If you want, I can:
- show the full top-20 table with exact numeric fields, or
- expand to the next-ranked pairs, or
- group anomalies by service to prioritize likely root causes.

The original code execution output of IPython Kernel is also provided below for reference:

(5298,                 cmdb_id                                           kpi_name  total_points_in_window  anomalies_high         earliest_high_ts  anomalies_low          earliest_low_ts  max_value_in_window  min_value_in_window    global_P95     global_P5
521         adservice-2         container.node-5.container_cpu_cfs_periods                      31               3  2022-03-21 05:39:00 CST              3  2022-03-21 05:35:00 CST         1.251667e+02         6.150000e+01  1.168333e+02  6.950000e+01
824          adservice2  runtime.java_lang_MemoryPool_Usage_used.Eden_S...                      31               3  2022-03-21 05:39:00 CST              3  2022-03-21 05:36:00 CST         5.468778e+06         3.638548e+06  5.265349e+06  3.839834e+06
850          adservice2   runtime.java_lang_OperatingSystem_ProcessCpuLoad                      31               3  2022-03-21 05:33:00 CST              3  2022-03-21 05:51:00 CST         3.433208e-03         1.797533e-04  4.021179e-04  1.902615e-04
852          adservice2    runtime.java_lang_OperatingSystem_SystemCpuLoad                      31               3  2022-03-21 05:42:00 CST              3  2022-03-21 05:30:00 CST         2.705409e-02         2.469559e-02  2.693639e-02  2.479791e-02
953        adservice2-0       container.node-6.container_cpu_usage_seconds                      31               3  2022-03-21 05:34:00 CST              3  2022-03-21 05:43:00 CST         4.691739e+00         1.516656e-01  3.957771e-01  1.605385e-01
954        adservice2-0        container.node-6.container_cpu_user_seconds                      31               3  2022-03-21 05:34:00 CST              3  2022-03-21 05:43:00 CST         4.750000e+00         1.300000e-01  3.600000e-01  1.300000e-01
1034       adservice2-0  mesh.destination.unknown.adservice2.istio_requ...                      31               3  2022-03-21 05:33:00 CST              3  2022-03-21 05:36:00 CST         1.596500e-01         1.257500e-01  1.387000e-01  1.282000e-01
1390     cartservice2-0  container.node-5.container_network_receive_MB....                      31               3  2022-03-21 05:30:00 CST              3  2022-03-21 05:41:00 CST         6.016920e-01         1.592541e-02  6.305408e-02  1.883698e-02
1659  checkoutservice-1  mesh.destination.frontend.checkoutservice.isti...                      31               3  2022-03-21 05:48:00 CST              3  2022-03-21 05:30:00 CST         1.962500e+03         0.000000e+00  1.570000e+03  3.925000e+02
1669  checkoutservice-1  mesh.destination.frontend.checkoutservice.isti...                      31               3  2022-03-21 05:48:00 CST              3  2022-03-21 05:30:00 CST         3.625000e+03         0.000000e+00  2.900000e+03  7.250000e+02
1673  checkoutservice-1  mesh.source.checkoutservice.cartservice.istio_...                      31               3  2022-03-21 05:48:00 CST              3  2022-03-21 05:30:00 CST         8.000000e+03         0.000000e+00  6.400000e+03  1.600000e+03
1676  checkoutservice-1  mesh.source.checkoutservice.cartservice.istio_...                      31               3  2022-03-21 05:48:00 CST              3  2022-03-21 05:30:00 CST         1.025000e-02         0.000000e+00  9.200000e-03  1.550000e-03
1683  checkoutservice-1  mesh.source.checkoutservice.cartservice.istio_...                      31               3  2022-03-21 05:48:00 CST              3  2022-03-21 05:30:00 CST         9.500000e+02         0.000000e+00  7.600000e+02  1.900000e+02
1686  checkoutservice-1  mesh.source.checkoutservice.currencyservice.is...                      31               3  2022-03-21 05:48:00 CST              3  2022-03-21 05:30:00 CST         1.162500e+04         0.000000e+00  9.300000e+03  2.325000e+03
1687  checkoutservice-1  mesh.source.checkoutservice.currencyservice.is...                      31               3  2022-03-21 05:48:00 CST              3  2022-03-21 05:30:00 CST         1.087500e-02         0.000000e+00  8.375000e-03  1.575000e-03
1690  checkoutservice-1  mesh.source.checkoutservice.currencyservice.is...                      31               3  2022-03-21 05:48:00 CST              3  2022-03-21 05:30:00 CST         1.687500e+03         0.000000e+00  1.350000e+03  3.375000e+02
1693  checkoutservice-1  mesh.source.checkoutservice.emailservice.istio...                      31               3  2022-03-21 05:48:00 CST              3  2022-03-21 05:30:00 CST         4.625000e+03         0.000000e+00  3.700000e+03  9.250000e+02
1695  checkoutservice-1  mesh.source.checkoutservice.emailservice.istio...                      31               3  2022-03-21 05:48:00 CST              3  2022-03-21 05:30:00 CST         5.125000e-03         0.000000e+00  4.100000e-03  1.025000e-03
1700  checkoutservice-1  mesh.source.checkoutservice.emailservice.istio...                      31               3  2022-03-21 05:48:00 CST              3  2022-03-21 05:30:00 CST         5.375000e+02         0.000000e+00  4.300000e+02  1.075000e+02
1702  checkoutservice-1  mesh.source.checkoutservice.jaeger-collector.i...                      31               3  2022-03-21 05:48:00 CST              3  2022-03-21 05:30:00 CST         3.690000e+04         0.000000e+00  2.810000e+04  7.025000e+03)```
```

In [ ]:
# In[3]:

# Reuse existing variables from the kernel: df_metric, thresholds, window
# If any are missing, recreate them (safe-guard). But per earlier steps they exist.

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')
start_window = 1647811800
end_window = 1647813600

# Ensure thresholds (global P95/P5 per (cmdb_id,kpi_name)) exists; if not, compute it.
if 'thresholds' not in globals():
    p95 = df_metric.groupby(['cmdb_id', 'kpi_name'])['value'].quantile(0.95).rename('global_P95')
    p5 = df_metric.groupby(['cmdb_id', 'kpi_name'])['value'].quantile(0.05).rename('global_P5')
    thresholds = pd.concat([p95, p5], axis=1).reset_index()

# Ensure window (rows in incident window merged with thresholds and anomaly flags) exists; if not, compute it.
if 'window' not in globals():
    window = df_metric[(df_metric['timestamp'] >= start_window) & (df_metric['timestamp'] <= end_window)].copy()
    window = window.merge(thresholds, on=['cmdb_id', 'kpi_name'], how='left')
    window['is_high'] = (window['value'] > window['global_P95']).astype(int)
    window['is_low'] = (window['value'] < window['global_P5']).astype(int)

# 1-3) Compute per-cmdb_id aggregates based on window rows and global thresholds
# a) per-cmdb totals from window rows
cmdb_window_agg = window.groupby('cmdb_id').agg(
    total_points_in_window = ('value', 'size'),
    total_anomalies_high = ('is_high', 'sum'),
    total_anomalies_low = ('is_low', 'sum'),
).reset_index()

# b) earliest anomaly timestamp per cmdb_id (consider both high and low)
anom_rows = window.loc[(window['is_high'] == 1) | (window['is_low'] == 1)].copy()
earliest_anom = anom_rows.groupby('cmdb_id')['timestamp'].min().rename('earliest_anomaly_ts').reset_index()

# c) distinct_kpis_with_anomalies: count of kpi_name per cmdb_id with at least one anomaly in window
kpianom = window.groupby(['cmdb_id','kpi_name']).agg(
    anomaly_count = (lambda x: ( (x['is_high'] == 1) | (x['is_low'] == 1) ).sum()) if False else ('is_high', 'sum') # placeholder; will recompute properly below
)

# Proper per-(cmdb_id,kpi_name) anomaly counts (sum of high+low)
pk = window.groupby(['cmdb_id','kpi_name']).agg(
    anomalies_high = ('is_high','sum'),
    anomalies_low = ('is_low','sum')
).reset_index()
pk['anomalies_total'] = pk['anomalies_high'] + pk['anomalies_low']

distinct_kpis_with_anomalies = (
    pk[pk['anomalies_total'] > 0]
    .groupby('cmdb_id')
    .size()
    .rename('distinct_kpis_with_anomalies')
    .reset_index()
)

# d) top_5_kpis_by_anomaly_count per cmdb_id (format as "kpi_name(count)" joined by ", ")
def top_kpis_for_cmdb(df_pk):
    # df_pk is subset for one cmdb_id
    df_sorted = df_pk.sort_values('anomalies_total', ascending=False)
    top = df_sorted.head(5)
    # format as "kpi_name(count)"
    items = top.apply(lambda r: f"{r['kpi_name']}({int(r['anomalies_total'])})", axis=1).tolist()
    return ", ".join(items)

top_kpis = (
    pk.sort_values(['cmdb_id','anomalies_total'], ascending=[True, False])
      .groupby('cmdb_id')
      .apply(lambda g: top_kpis_for_cmdb(g))
      .rename('top_5_kpis_by_anomaly_count')
      .reset_index()
)

# e) Combine per-cmdb aggregates; include cmdb_ids that have definitions in thresholds even if zero window points
all_cmdbs = pd.DataFrame({'cmdb_id': thresholds['cmdb_id'].unique()})

cmdbs = all_cmdbs.merge(cmdb_window_agg, on='cmdb_id', how='left')
cmdbs = cmdbs.merge(distinct_kpis_with_anomalies, on='cmdb_id', how='left')
cmdbs = cmdbs.merge(top_kpis, on='cmdb_id', how='left')
cmdbs = cmdbs.merge(earliest_anom, on='cmdb_id', how='left')

# fill NaNs with zeros or None appropriately
cmdbs['total_points_in_window'] = cmdbs['total_points_in_window'].fillna(0).astype(int)
cmdbs['total_anomalies_high'] = cmdbs['total_anomalies_high'].fillna(0).astype(int)
cmdbs['total_anomalies_low'] = cmdbs['total_anomalies_low'].fillna(0).astype(int)
cmdbs['anomalies_total'] = cmdbs['total_anomalies_high'] + cmdbs['total_anomalies_low']
cmdbs['distinct_kpis_with_anomalies'] = cmdbs['distinct_kpis_with_anomalies'].fillna(0).astype(int)
cmdbs['top_5_kpis_by_anomaly_count'] = cmdbs['top_5_kpis_by_anomaly_count'].fillna('')  # empty string if none

# format earliest_anomaly_ts to Asia/Shanghai local string, keep None if NaN
def fmt_ts_nullable(x):
    if pd.isna(x):
        return None
    return pd.to_datetime(int(x), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')

cmdbs['earliest_anomaly_ts'] = cmdbs['earliest_anomaly_ts'].apply(fmt_ts_nullable)

# 4) Prepare outputs
distinct_cmdb_count = cmdbs['cmdb_id'].nunique()
cmdbs_with_anomalies_count = (cmdbs['anomalies_total'] > 0).sum()

# Decide sorting: if no anomalies anywhere, sort by total_points_in_window desc; else by anomalies_total desc
if cmdbs['anomalies_total'].sum() == 0:
    sort_by = ['total_points_in_window', 'cmdb_id']
    asc = [False, True]
else:
    sort_by = ['anomalies_total', 'total_points_in_window', 'cmdb_id']
    asc = [False, False, True]

top20_cmdbs = cmdbs.sort_values(sort_by, ascending=asc).head(20)

# Select and order requested columns
top20_summary = top20_cmdbs[[
    'cmdb_id',
    'total_points_in_window',
    'anomalies_total',
    'total_anomalies_high',
    'total_anomalies_low',
    'distinct_kpis_with_anomalies',
    'earliest_anomaly_ts',
    'top_5_kpis_by_anomaly_count'
]]

# Final compact outputs (variables for IPython display)
distinct_cmdb_count, cmdbs_with_anomalies_count, top20_summary

```
Out[3]:
```
Summary of cmdb_id-level anomaly aggregation (incident window 2022-03-21 05:30–06:00 CST / timestamps 1647811800–1647813600):

- Distinct cmdb_id processed: 59
- cmdb_id with anomalies_total > 0: 59

Top 20 cmdb_id by anomalies_total (cmdb_id: total_points_in_window, anomalies_total (high, low), distinct_kpis_with_anomalies, earliest_anomaly_ts, top_5_kpis_by_anomaly_count):

1. checkoutservice-1: 4464, anomalies_total=285 (high=134, low=151), distinct_kpis=60, earliest=2022-03-21 05:30:00 CST, top_kpis e.g. mesh.destination.frontend.checkoutservice.isti...
2. frontend-2: 5828, anomalies_total=235 (high=164, low=71), distinct_kpis=60, earliest=2022-03-21 05:33:00 CST, top_kpis e.g. mesh.destination.unknown.frontend.istio_respon...
3. frontend-1: 5828, anomalies_total=225 (high=185, low=40), distinct_kpis=66, earliest=2022-03-21 05:30:00 CST, top_kpis e.g. mesh.source.frontend.checkoutservice.istio_req...
4. frontend-0: 5828, anomalies_total=202 (high=163, low=39), distinct_kpis=61, earliest=2022-03-21 05:32:00 CST, top_kpis e.g. mesh.destination.unknown.frontend.istio_reques...
5. adservice: 10571, anomalies_total=146 (high=123, low=23), distinct_kpis=51, earliest=2022-03-21 05:30:00 CST, top_kpis e.g. app.http.mrt(5), app.grpc.mrt(5), runtime.jvm_...
6. productcatalogservice-0: 3720, anomalies_total=136 (high=96, low=40), distinct_kpis=38, earliest=2022-03-21 05:31:00 CST, top_kpis e.g. mesh.source.productcatalogservice.jaeger-colle...
7. productcatalogservice-2: 3596, anomalies_total=133 (high=101, low=32), distinct_kpis=38, earliest=2022-03-21 05:32:00 CST, top_kpis e.g. container.node-6.container_cpu_system_seconds(...
8. checkoutservice-2: 4340, anomalies_total=126 (high=117, low=9), distinct_kpis=63, earliest=2022-03-21 05:30:00 CST, top_kpis e.g. container.node-5.container_memory_working_set_...
9. productcatalogservice-1: 3720, anomalies_total=124 (high=90, low=34), distinct_kpis=37, earliest=2022-03-21 05:31:00 CST, top_kpis e.g. container.node-6.container_cpu_cfs_periods(5),...
10. adservice2: 10323, anomalies_total=122 (high=84, low=38), distinct_kpis=38, earliest=2022-03-21 05:30:00 CST, top_kpis e.g. runtime.java_lang_MemoryPool_Usage_used.Eden_S...
11. currencyservice-2: 2728, anomalies_total=121 (high=79, low=42), distinct_kpis=32, earliest=2022-03-21 05:30:00 CST, top_kpis e.g. container.node-6.container_cpu_cfs_periods(6),...
12. cartservice-0: 3100, anomalies_total=102 (high=80, low=22), distinct_kpis=30, earliest=2022-03-21 05:31:00 CST, top_kpis e.g. mesh.destination.frontend.cartservice.istio_re...
13. node-5: 1748, anomalies_total=95 (high=60, low=35), distinct_kpis=34, earliest=2022-03-21 05:30:00 CST, top_kpis e.g. system.cpu.user(6), system.cpu.pct_usage(5), s...
14. currencyservice-0: 2480, anomalies_total=94 (high=66, low=28), distinct_kpis=31, earliest=2022-03-21 05:31:00 CST, top_kpis e.g. container.node-6.container_cpu_user_seconds(5)...
15. node-3: 1748, anomalies_total=93 (high=58, low=35), distinct_kpis=33, earliest=2022-03-21 05:30:00 CST, top_kpis e.g. system.mem.free(6), system.net.bytes_rcvd(6), ...
16. cartservice-1: 2976, anomalies_total=89 (high=59, low=30), distinct_kpis=33, earliest=2022-03-21 05:32:00 CST, top_kpis e.g. mesh.destination.frontend.cartservice.istio_re...
17. node-2: 1779, anomalies_total=87 (high=53, low=34), distinct_kpis=29, earliest=2022-03-21 05:32:00 CST, top_kpis e.g. system.net.bytes_sent(5), system.net.tcp.in_se...
18. node-4: 1707, anomalies_total=83 (high=60, low=23), distinct_kpis=30, earliest=2022-03-21 05:30:00 CST, top_kpis e.g. system.net.tcp.out_segs(5), system.cpu.iowait(...
19. cartservice-2: 3100, anomalies_total=79 (high=67, low=12), distinct_kpis=33, earliest=2022-03-21 05:30:00 CST, top_kpis e.g. container.node-6.container_cpu_cfs_periods(4),...
20. recommendationservice-1: 3038, anomalies_total=79 (high=63, low=16), distinct_kpis=24, earliest=2022-03-21 05:30:00 CST, top_kpis e.g. container.node-6.container_cpu_usage_seconds(4)...

Interpretation notes (concise):
- All 59 processed cmdb_id show anomalies during the incident window; many services (checkoutservice, frontend instances, adservice, productcatalogservice variants, cartservice, currencyservice) and several nodes show correlated anomalies across many KPIs.
- The largest anomaly counts are concentrated in checkoutservice-1 and frontend pods, followed by ad/productcatalog/checkouts/cart services and some nodes. Many affected cmdb_id have dozens of distinct KPIs with anomalies, indicating broad/related deviations rather than single-metric spikes.
- Earliest anomalies appear around 2022-03-21 05:30:00 CST for many top cmdb_id.

If you want, I can: provide the exact top-20 table with full KPI names and counts, or group totals by service (aggregating pods) to prioritize likely root causes.

The original code execution output of IPython Kernel is also provided below for reference:

(59, np.int64(59),                     cmdb_id  total_points_in_window  anomalies_total  total_anomalies_high  total_anomalies_low  distinct_kpis_with_anomalies      earliest_anomaly_ts                        top_5_kpis_by_anomaly_count
13        checkoutservice-1                    4464              285                   134                  151                            60  2022-03-21 05:30:00 CST  mesh.destination.frontend.checkoutservice.isti...
29               frontend-2                    5828              235                   164                   71                            60  2022-03-21 05:33:00 CST  mesh.destination.unknown.frontend.istio_respon...
28               frontend-1                    5828              225                   185                   40                            66  2022-03-21 05:30:00 CST  mesh.source.frontend.checkoutservice.istio_req...
27               frontend-0                    5828              202                   163                   39                            61  2022-03-21 05:32:00 CST  mesh.destination.unknown.frontend.istio_reques...
0                 adservice                   10571              146                   123                   23                            51  2022-03-21 05:30:00 CST  app.http.mrt(5), app.grpc.mrt(5), runtime.jvm_...
43  productcatalogservice-0                    3720              136                    96                   40                            38  2022-03-21 05:31:00 CST  mesh.source.productcatalogservice.jaeger-colle...
45  productcatalogservice-2                    3596              133                   101                   32                            38  2022-03-21 05:32:00 CST  container.node-6.container_cpu_system_seconds(...
14        checkoutservice-2                    4340              126                   117                    9                            63  2022-03-21 05:30:00 CST  container.node-5.container_memory_working_set_...
44  productcatalogservice-1                    3720              124                    90                   34                            37  2022-03-21 05:31:00 CST  container.node-6.container_cpu_cfs_periods(5),...
4                adservice2                   10323              122                    84                   38                            38  2022-03-21 05:30:00 CST  runtime.java_lang_MemoryPool_Usage_used.Eden_S...
19        currencyservice-2                    2728              121                    79                   42                            32  2022-03-21 05:30:00 CST  container.node-6.container_cpu_cfs_periods(6),...
7             cartservice-0                    3100              102                    80                   22                            30  2022-03-21 05:31:00 CST  mesh.destination.frontend.cartservice.istio_re...
35                   node-5                    1748               95                    60                   35                            34  2022-03-21 05:30:00 CST  system.cpu.user(6), system.cpu.pct_usage(5), s...
17        currencyservice-0                    2480               94                    66                   28                            31  2022-03-21 05:31:00 CST  container.node-6.container_cpu_user_seconds(5)...
33                   node-3                    1748               93                    58                   35                            33  2022-03-21 05:30:00 CST  system.mem.free(6), system.net.bytes_rcvd(6), ...
8             cartservice-1                    2976               89                    59                   30                            33  2022-03-21 05:32:00 CST  mesh.destination.frontend.cartservice.istio_re...
32                   node-2                    1779               87                    53                   34                            29  2022-03-21 05:32:00 CST  system.net.bytes_sent(5), system.net.tcp.in_se...
34                   node-4                    1707               83                    60                   23                            30  2022-03-21 05:30:00 CST  system.net.tcp.out_segs(5), system.cpu.iowait(...
9             cartservice-2                    3100               79                    67                   12                            33  2022-03-21 05:30:00 CST  container.node-6.container_cpu_cfs_periods(4),...
49  recommendationservice-1                    3038               79                    63                   16                            24  2022-03-21 05:30:00 CST  container.node-6.container_cpu_usage_seconds(4...)```
```

In [ ]:
# In[4]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')
start_window = 1647811800
end_window = 1647813600

# Ensure df_trace is available (it was loaded earlier); if not, load it.
try:
    df_trace
except NameError:
    df_trace = pd.read_csv('trace.csv')

# 1) Compute global P95 and P5 per (cmdb_id, trace_name) using full series
if df_trace.empty:
    thresholds_trace = pd.DataFrame(columns=['cmdb_id','trace_name','global_P95','global_P5'])
else:
    p95 = df_trace.groupby(['cmdb_id','trace_name'])['value'].quantile(0.95).rename('global_P95')
    p5 = df_trace.groupby(['cmdb_id','trace_name'])['value'].quantile(0.05).rename('global_P5')
    thresholds_trace = pd.concat([p95, p5], axis=1).reset_index()

# 2) Filter trace rows to incident window
window_trace = df_trace[(df_trace['timestamp'] >= start_window) & (df_trace['timestamp'] <= end_window)].copy()

# 3) For each (cmdb_id, trace_name) in the window compute required fields
if window_trace.empty:
    # Prepare empty results
    distinct_pairs = 0
    top20_summary = pd.DataFrame(columns=[
        'cmdb_id','trace_name','total_points_in_window','anomalies_total','anomalies_high','earliest_high_ts',
        'anomalies_low','earliest_low_ts','max_value_in_window','min_value_in_window','global_P95','global_P5'
    ])
else:
    # Merge global thresholds into window rows
    window_trace = window_trace.merge(thresholds_trace, on=['cmdb_id','trace_name'], how='left')
    # Flags
    window_trace['is_high'] = (window_trace['value'] > window_trace['global_P95']).astype(int)
    window_trace['is_low'] = (window_trace['value'] < window_trace['global_P5']).astype(int)
    # Aggregations
    grp = window_trace.groupby(['cmdb_id','trace_name'])
    agg = grp.agg(
        total_points_in_window = ('value','size'),
        anomalies_high = ('is_high','sum'),
        anomalies_low = ('is_low','sum'),
        max_value_in_window = ('value','max'),
        min_value_in_window = ('value','min'),
        # keep global thresholds by taking first (they are same per group)
        global_P95 = ('global_P95','first'),
        global_P5 = ('global_P5','first')
    ).reset_index()
    agg['anomalies_total'] = agg['anomalies_high'] + agg['anomalies_low']
    # earliest timestamps for high/low
    earliest_high = window_trace.loc[window_trace['is_high'] == 1].groupby(['cmdb_id','trace_name'])['timestamp'].min().rename('earliest_high_ts').reset_index()
    earliest_low = window_trace.loc[window_trace['is_low'] == 1].groupby(['cmdb_id','trace_name'])['timestamp'].min().rename('earliest_low_ts').reset_index()
    agg = agg.merge(earliest_high, on=['cmdb_id','trace_name'], how='left')
    agg = agg.merge(earliest_low, on=['cmdb_id','trace_name'], how='left')
    # format earliest timestamps to Asia/Shanghai local strings, keep None where missing
    def fmt_ts_nullable(x):
        if pd.isna(x):
            return None
        return pd.to_datetime(int(x), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
    agg['earliest_high_ts'] = agg['earliest_high_ts'].apply(fmt_ts_nullable)
    agg['earliest_low_ts'] = agg['earliest_low_ts'].apply(fmt_ts_nullable)
    # 4) Choose top 20 by anomalies_total desc; if no anomalies anywhere, top by total_points_in_window
    distinct_pairs = agg.shape[0]
    if agg['anomalies_total'].sum() == 0:
        top20_summary = agg.sort_values(['total_points_in_window','cmdb_id','trace_name'], ascending=[False,True,True]).head(20)
    else:
        top20_summary = agg.sort_values(['anomalies_total','total_points_in_window','cmdb_id','trace_name'], ascending=[False,False,True,True]).head(20)
    # select and order requested columns
    top20_summary = top20_summary[[
        'cmdb_id','trace_name','total_points_in_window','anomalies_total','anomalies_high','earliest_high_ts',
        'anomalies_low','earliest_low_ts','max_value_in_window','min_value_in_window','global_P95','global_P5'
    ]].reset_index(drop=True)

# Final compact outputs for display
distinct_pairs, top20_summary

```
Out[4]:
```
```python
# Plain-English summary of trace-level anomaly aggregation (compact)
summary = (
    "Trace-level anomaly summary:\n\n"
    "- Distinct (cmdb_id, trace_name) pairs processed in the window: 1,232.\n"
    "- Many trace pairs show anomalies; the table was sorted by anomalies_total and the top results have anomalies_total = 4.\n"
    "- Top pairs (examples): adservice-0 (trace.from_frontend-0.duration_p95), adservice-0 (trace.from_frontend-2.duration_p95), "
    "adservice-2 (trace.from_frontend-2.duration_p95), cartservice-* (trace.self.row_count and trace.from_frontend.*.row_count), and several currencyservice trace metrics.\n"
    "- Typical pattern for top pairs: total_points_in_window = 30, anomalies_total = 4 (often anomalies_high = 2 and anomalies_low = 2).\n"
    "- Earliest anomaly timestamps for these top pairs range roughly between 2022-03-21 05:30:00 CST and 05:55:00 CST (Asia/Shanghai timezone).\n"
    "- Max/min and global thresholds vary by trace metric (examples shown in the table); anomalies are small in count per pair but appear across many trace metrics and services, indicating widespread but modest deviations during the incident window.\n\n"
    "Interpretation / next steps:\n"
    "- The anomalies are distributed across many services (adservice, cartservice, currencyservice, etc.), suggesting correlated latency/trace behavior rather than isolated single-trace failures.\n"
    "- If you want, I can: show the full top-20 table with exact numeric fields, aggregate anomalies by service, or correlate these trace anomalies with the metric-level anomalies already computed."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(1232,               cmdb_id                           trace_name  total_points_in_window  anomalies_total  anomalies_high         earliest_high_ts  anomalies_low          earliest_low_ts  max_value_in_window  min_value_in_window  global_P95  global_P5
0         adservice-0   trace.from_frontend-0.duration_p95                      30                4               2  2022-03-21 05:30:00 CST              2  2022-03-21 05:38:00 CST             0.000074             0.000017    0.000056   0.000019
1         adservice-0   trace.from_frontend-2.duration_p95                      30                4               2  2022-03-21 05:41:00 CST              2  2022-03-21 05:51:00 CST             0.000072             0.000016    0.000038   0.000017
2         adservice-2   trace.from_frontend-2.duration_p95                      30                4               2  2022-03-21 05:34:00 CST              2  2022-03-21 05:30:00 CST             0.000070             0.000013    0.000028   0.000013
3       cartservice-0                 trace.self.row_count                      30                4               2  2022-03-21 05:55:00 CST              2  2022-03-21 05:32:00 CST            55.000000            27.000000   49.650000  28.900000
4       cartservice-1      trace.from_frontend-0.row_count                      30                4               2  2022-03-21 05:52:00 CST              2  2022-03-21 05:32:00 CST            18.000000             9.000000   17.550000   9.900000
5       cartservice-1                 trace.self.row_count                      30                4               2  2022-03-21 05:55:00 CST              2  2022-03-21 05:32:00 CST            54.000000            25.000000   50.100000  27.350000
6       cartservice-2      trace.from_frontend-1.row_count                      30                4               2  2022-03-21 05:44:00 CST              2  2022-03-21 05:32:00 CST            18.000000            10.000000   17.550000  10.450000
7       cartservice-2                 trace.self.row_count                      30                4               2  2022-03-21 05:55:00 CST              2  2022-03-21 05:32:00 CST            54.000000            26.000000   50.100000  28.900000
8   currencyservice-0  trace.from_frontend-0.duration_mean                      30                4               2  2022-03-21 05:48:00 CST              2  2022-03-21 05:32:00 CST             0.000148             0.000112    0.000143   0.000113
9   currencyservice-0      trace.from_frontend-0.row_count                      30                4               2  2022-03-21 05:37:00 CST              2  2022-03-21 05:33:00 CST            68.000000            24.000000   62.100000  31.450000
10  currencyservice-0  trace.from_frontend-1.duration_mean                      30                4               2  2022-03-21 05:51:00 CST              2  2022-03-21 05:37:00 CST             0.000145             0.000109    0.000142   0.000111
11  currencyservice-0   trace.from_frontend-1.duration_p95                      30                4               2  2022-03-21 05:47:00 CST              2  2022-03-21 05:31:00 CST             0.000165             0.000129    0.000163   0.000135
12  currencyservice-0      trace.from_frontend-1.row_count                      30                4               2  2022-03-21 05:44:00 CST              2  2022-03-21 05:30:00 CST            65.000000            32.000000   62.550000  34.450000
13  currencyservice-0  trace.from_frontend-2.duration_mean                      30                4               2  2022-03-21 05:33:00 CST              2  2022-03-21 05:32:00 CST             0.000169             0.000104    0.000138   0.000106
14  currencyservice-0   trace.from_frontend-2.duration_p95                      30                4               2  2022-03-21 05:33:00 CST              2  2022-03-21 05:32:00 CST             0.000353             0.000120    0.000170   0.000126
15  currencyservice-1  trace.from_frontend-0.duration_mean                      30                4               2  2022-03-21 05:40:00 CST              2  2022-03-21 05:30:00 CST             0.000146             0.000115    0.000143   0.000115
16  currencyservice-1   trace.from_frontend-0.duration_p95                      30                4               2  2022-03-21 05:45:00 CST              2  2022-03-21 05:30:00 CST             0.000175             0.000135    0.000168   0.000140
17  currencyservice-1   trace.from_frontend-1.duration_p95                      30                4               2  2022-03-21 05:30:00 CST              2  2022-03-21 05:33:00 CST             0.000165             0.000137    0.000163   0.000140
18  currencyservice-1      trace.from_frontend-1.row_count                      30                4               2  2022-03-21 05:44:00 CST              2  2022-03-21 05:30:00 CST            64.000000            32.000000   62.550000  34.450000
19  currencyservice-1  trace.from_frontend-2.duration_mean                      30                4               2  2022-03-21 05:57:00 CST              2  2022-03-21 05:32:00 CST             0.000173             0.000109    0.000161   0.000112)```
```

In [ ]:
# In[5]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')
start_window = 1647811800
end_window = 1647813600

# ensure df_log is available
try:
    df_log
except NameError:
    df_log = pd.read_csv('log.csv')

# 1) global P95 and P5 per (cmdb_id, log_name) using full series (do NOT filter first)
if df_log.empty:
    thresholds_log = pd.DataFrame(columns=['cmdb_id','log_name','global_P95','global_P5'])
else:
    p95 = df_log.groupby(['cmdb_id','log_name'])['value'].quantile(0.95).rename('global_P95')
    p5 = df_log.groupby(['cmdb_id','log_name'])['value'].quantile(0.05).rename('global_P5')
    thresholds_log = pd.concat([p95, p5], axis=1).reset_index()

# 2) filter to incident window
window_log = df_log[(df_log['timestamp'] >= start_window) & (df_log['timestamp'] <= end_window)].copy()

# 3) compute per-(cmdb_id, log_name) stats within window, using global thresholds
if window_log.empty:
    pair_count = 0
    top20_pairs = pd.DataFrame(columns=[
        'cmdb_id','log_name','total_points_in_window','anomalies_total','anomalies_high','earliest_high_ts',
        'anomalies_low','earliest_low_ts','max_value_in_window','min_value_in_window','global_P95','global_P5'
    ])
    top20_cmdbs = pd.DataFrame(columns=['cmdb_id','total_log_metrics','cmdb_anomalies_total','earliest_log_anomaly_ts'])
else:
    # merge thresholds into window rows
    window_log = window_log.merge(thresholds_log, on=['cmdb_id','log_name'], how='left')
    window_log['is_high'] = (window_log['value'] > window_log['global_P95']).astype(int)
    window_log['is_low'] = (window_log['value'] < window_log['global_P5']).astype(int)
    
    # aggregate per pair
    grp = window_log.groupby(['cmdb_id','log_name'])
    pair_agg = grp.agg(
        total_points_in_window = ('value','size'),
        anomalies_high = ('is_high','sum'),
        anomalies_low = ('is_low','sum'),
        max_value_in_window = ('value','max'),
        min_value_in_window = ('value','min'),
        global_P95 = ('global_P95','first'),
        global_P5 = ('global_P5','first')
    ).reset_index()
    pair_agg['anomalies_total'] = pair_agg['anomalies_high'] + pair_agg['anomalies_low']
    
    # earliest high/low timestamps per pair
    earliest_high = window_log.loc[window_log['is_high']==1].groupby(['cmdb_id','log_name'])['timestamp'].min().rename('earliest_high_ts').reset_index()
    earliest_low = window_log.loc[window_log['is_low']==1].groupby(['cmdb_id','log_name'])['timestamp'].min().rename('earliest_low_ts').reset_index()
    pair_agg = pair_agg.merge(earliest_high, on=['cmdb_id','log_name'], how='left')
    pair_agg = pair_agg.merge(earliest_low, on=['cmdb_id','log_name'], how='left')
    
    # format earliest timestamps to Asia/Shanghai strings
    def fmt_ts_nullable(x):
        if pd.isna(x):
            return None
        return pd.to_datetime(int(x), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
    pair_agg['earliest_high_ts'] = pair_agg['earliest_high_ts'].apply(fmt_ts_nullable)
    pair_agg['earliest_low_ts'] = pair_agg['earliest_low_ts'].apply(fmt_ts_nullable)
    
    # 4a) Top 20 pairs sorted by anomalies_total desc (or by total_points if no anomalies)
    pair_count = pair_agg.shape[0]
    if pair_agg['anomalies_total'].sum() == 0:
        top20_pairs = pair_agg.sort_values(['total_points_in_window','cmdb_id','log_name'], ascending=[False,True,True]).head(20)
    else:
        top20_pairs = pair_agg.sort_values(['anomalies_total','total_points_in_window','cmdb_id','log_name'], ascending=[False,False,True,True]).head(20)
    
    top20_pairs = top20_pairs[[
        'cmdb_id','log_name','total_points_in_window','anomalies_total','anomalies_high','earliest_high_ts',
        'anomalies_low','earliest_low_ts','max_value_in_window','min_value_in_window','global_P95','global_P5'
    ]].reset_index(drop=True)
    
    # 4b) Aggregate by cmdb_id
    # total_log_metrics: number of distinct log_name seen in-window
    total_log_metrics = window_log.groupby('cmdb_id')['log_name'].nunique().rename('total_log_metrics').reset_index()
    # cmdb_anomalies_total: sum anomalies_total across log metrics
    pair_anom_sum = pair_agg.groupby('cmdb_id')['anomalies_total'].sum().rename('cmdb_anomalies_total').reset_index()
    # earliest_log_anomaly_ts: min timestamp of any log anomaly (high or low) per cmdb_id
    anom_rows = window_log.loc[(window_log['is_high']==1) | (window_log['is_low']==1)].copy()
    if not anom_rows.empty:
        earliest_log_anom = anom_rows.groupby('cmdb_id')['timestamp'].min().rename('earliest_log_anomaly_ts').reset_index()
        earliest_log_anom['earliest_log_anomaly_ts'] = earliest_log_anom['earliest_log_anomaly_ts'].apply(fmt_ts_nullable)
    else:
        earliest_log_anom = pd.DataFrame(columns=['cmdb_id','earliest_log_anomaly_ts'])
    
    cmdb_agg = total_log_metrics.merge(pair_anom_sum, on='cmdb_id', how='left').merge(earliest_log_anom, on='cmdb_id', how='left')
    cmdb_agg['cmdb_anomalies_total'] = cmdb_agg['cmdb_anomalies_total'].fillna(0).astype(int)
    cmdb_agg['earliest_log_anomaly_ts'] = cmdb_agg['earliest_log_anomaly_ts'].where(~cmdb_agg['earliest_log_anomaly_ts'].isna(), None)
    
    # sort and limit top 20 cmdb_id by cmdb_anomalies_total desc (or by total_log_metrics if no anomalies)
    if cmdb_agg['cmdb_anomalies_total'].sum() == 0:
        top20_cmdbs = cmdb_agg.sort_values(['total_log_metrics','cmdb_id'], ascending=[False,True]).head(20)
    else:
        top20_cmdbs = cmdb_agg.sort_values(['cmdb_anomalies_total','total_log_metrics','cmdb_id'], ascending=[False,False,True]).head(20)
    
    top20_cmdbs = top20_cmdbs[['cmdb_id','total_log_metrics','cmdb_anomalies_total','earliest_log_anomaly_ts']].reset_index(drop=True)

# Final compact outputs: pair_count, top20_pairs, top20_cmdbs
pair_count, top20_pairs, top20_cmdbs

```
Out[5]:
```
- Distinct (cmdb_id, log_name) pairs seen in-window: 62.

Top findings (log-level pairs):
- Top 20 (cmdb_id, log_name) are dominated by log.row_count for many services and all show anomalies_total = 4 (typically anomalies_high = 2 and anomalies_low = 2).
- Typical per-pair window size = 30–31 points (most = 31).
- Earliest anomaly times for top pairs fall in ~2022-03-21 05:32:00 CST → 2022-03-21 05:56:00 CST (Asia/Shanghai).
- Example rows from the top 20:
  - cartservice-0, log.row_count — 31 points, anomalies_total=4 (high=2 @ 05:55, low=2 @ 05:32), max=481, min=9, global_P95≈464.50, global_P5≈254.50
  - frontend-0, log.row_count — 31 points, anomalies_total=4 (high=2 @ 05:52, low=2 @ 05:36), max=970, min=2, global_P95≈898.50, global_P5≈499.00
  - currencyservice-0, log.row_count — 31 points, anomalies_total=4 (high=2 @ 05:56, low=2 @ 05:36), max=427, min=11, global_P95≈417.50, global_P5≈213.00
  (other top pairs follow the same pattern.)

CMDB-level aggregates:
- Number of cmdb_id with log metrics in-window reported in the aggregate table: 20 shown (full set contains more).
- Many cmdb_id have exactly 2 distinct log metrics in-window and cmdb_anomalies_total = 4 (i.e., the two metrics together produce 4 anomalies).
- Examples: adservice-1, cartservice-0/1/2, currencyservice-0/1/2, frontend-0/1/2, productcatalogservice-*, recommendationservice-*, shippingservice-* — each shows total_log_metrics = 2 and cmdb_anomalies_total = 4, with earliest_log_anomaly_ts typically at ~05:31–05:36 CST.

Interpretation (brief):
- Log counts show small but consistent anomalies across many services in the incident window — mostly 2 high and 2 low deviations per log metric.
- Anomalies are concurrent across many services (cartservice, frontend, currencyservice, productcatalog, recommendation, shipping), with earliest occurrences clustering around 05:30–05:36 CST and some high events later (~05:52–05:56 CST).
- This pattern suggests a system-wide / cross-service perturbation during the incident window rather than isolated single-log anomalies.

If you want, I can: show the full top-20 tables, expand cmdb-level results beyond 20 rows, or correlate these log anomalies with metric/trace anomalies already computed.

The original code execution output of IPython Kernel is also provided below for reference:

(62,                     cmdb_id       log_name  total_points_in_window  anomalies_total  anomalies_high         earliest_high_ts  anomalies_low          earliest_low_ts  max_value_in_window  min_value_in_window  global_P95  global_P5
0             cartservice-0  log.row_count                      31                4               2  2022-03-21 05:55:00 CST              2  2022-03-21 05:32:00 CST                481.0                  9.0      464.50     254.50
1             cartservice-1  log.row_count                      31                4               2  2022-03-21 05:55:00 CST              2  2022-03-21 05:32:00 CST                490.0                  9.0      444.00     236.50
2             cartservice-2  log.row_count                      31                4               2  2022-03-21 05:52:00 CST              2  2022-03-21 05:32:00 CST                480.0                 18.0      444.50     254.00
3         currencyservice-0  log.row_count                      31                4               2  2022-03-21 05:56:00 CST              2  2022-03-21 05:36:00 CST                427.0                 11.0      417.50     213.00
4         currencyservice-1  log.row_count                      31                4               2  2022-03-21 05:56:00 CST              2  2022-03-21 05:36:00 CST                430.0                 12.0      416.50     212.50
5         currencyservice-2  log.row_count                      31                4               2  2022-03-21 05:56:00 CST              2  2022-03-21 05:36:00 CST                426.0                 12.0      413.00     208.00
6                frontend-0  log.row_count                      31                4               2  2022-03-21 05:52:00 CST              2  2022-03-21 05:36:00 CST                970.0                  2.0      898.50     499.00
7                frontend-1  log.row_count                      31                4               2  2022-03-21 05:44:00 CST              2  2022-03-21 05:34:00 CST                945.0                 12.0      894.00     543.00
8   productcatalogservice-2  log.row_count                      31                4               2  2022-03-21 05:55:00 CST              2  2022-03-21 05:32:00 CST                573.0                 54.0      513.50     302.50
9   recommendationservice-0  log.row_count                      31                4               2  2022-03-21 05:55:00 CST              2  2022-03-21 05:32:00 CST                119.0                  1.0      115.50      69.50
10  recommendationservice-2  log.row_count                      31                4               2  2022-03-21 05:55:00 CST              2  2022-03-21 05:32:00 CST                125.0                  1.0      120.00      65.50
11        shippingservice-2  log.row_count                      31                4               2  2022-03-21 05:52:00 CST              2  2022-03-21 05:32:00 CST                 72.0                  4.0       57.00      24.50
12              adservice-1  log.row_count                      30                4               2  2022-03-21 05:41:00 CST              2  2022-03-21 05:32:00 CST                 58.0                 28.0       57.10      29.80
13               frontend-2  log.row_count                      30                4               2  2022-03-21 05:53:00 CST              2  2022-03-21 05:32:00 CST                484.0                146.0      441.40     169.15
14  productcatalogservice-0  log.row_count                      30                4               2  2022-03-21 05:55:00 CST              2  2022-03-21 05:32:00 CST                553.0                281.0      539.40     316.50
15  productcatalogservice-1  log.row_count                      30                4               2  2022-03-21 05:55:00 CST              2  2022-03-21 05:32:00 CST                569.0                293.0      535.55     314.90
16  recommendationservice-1  log.row_count                      30                4               2  2022-03-21 05:55:00 CST              2  2022-03-21 05:32:00 CST                122.0                 62.0      116.10      71.45
17        shippingservice-0  log.row_count                      30                4               2  2022-03-21 05:52:00 CST              2  2022-03-21 05:31:00 CST                 72.0                 22.0       59.40      24.35
18        shippingservice-1  log.row_count                      30                4               2  2022-03-21 05:52:00 CST              2  2022-03-21 05:33:00 CST                 73.0                 25.0       55.55      27.90
19           emailservice-2  log.row_count                      29                4               2  2022-03-21 05:52:00 CST              2  2022-03-21 05:31:00 CST                 11.0                  1.0        9.60       1.40,                     cmdb_id  total_log_metrics  cmdb_anomalies_total  earliest_log_anomaly_ts
0               adservice-1                  2                     4  2022-03-21 05:32:00 CST
1             cartservice-0                  2                     4  2022-03-21 05:32:00 CST
2             cartservice-1                  2                     4  2022-03-21 05:32:00 CST
3             cartservice-2                  2                     4  2022-03-21 05:32:00 CST
4         currencyservice-0                  2                     4  2022-03-21 05:36:00 CST
5         currencyservice-1                  2                     4  2022-03-21 05:36:00 CST
6         currencyservice-2                  2                     4  2022-03-21 05:36:00 CST
7            emailservice-2                  2                     4  2022-03-21 05:31:00 CST
8                frontend-0                  2                     4  2022-03-21 05:36:00 CST
9                frontend-1                  2                     4  2022-03-21 05:34:00 CST
10               frontend-2                  2                     4  2022-03-21 05:32:00 CST
11  productcatalogservice-0                  2                     4  2022-03-21 05:32:00 CST
12  productcatalogservice-1                  2                     4  2022-03-21 05:32:00 CST
13  productcatalogservice-2                  2                     4  2022-03-21 05:32:00 CST
14  recommendationservice-0                  2                     4  2022-03-21 05:32:00 CST
15  recommendationservice-1                  2                     4  2022-03-21 05:32:00 CST
16  recommendationservice-2                  2                     4  2022-03-21 05:32:00 CST
17        shippingservice-0                  2                     4  2022-03-21 05:31:00 CST
18        shippingservice-1                  2                     4  2022-03-21 05:33:00 CST
19        shippingservice-2                  2                     4  2022-03-21 05:32:00 CST)```
```

In [ ]:
# In[6]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse cmdbs computed earlier (per-cmdb_id aggregates)
# Ensure cmdbs exists
try:
    cmdbs
except NameError:
    raise RuntimeError("cmdbs variable not found in kernel. Please run the previous cmdb-level aggregation step first.")

# 1) Build service name list (base services + "2" variants)
base_services = [
    "frontend","shippingservice","checkoutservice","currencyservice",
    "adservice","emailservice","cartservice","productcatalogservice",
    "recommendationservice","paymentservice"
]
service_names = base_services + [s + "2" for s in base_services]
# sort by length desc to ensure longest-prefix matching first (e.g., adservice2 before adservice)
service_names = sorted(service_names, key=lambda x: -len(x))

# helper to map cmdb_id -> service_name
def map_to_service(cmdb_id):
    # nodes map to themselves like node-1..node-6
    if cmdb_id.startswith('node-'):
        return cmdb_id  # keep as 'node-#'
    for svc in service_names:
        if cmdb_id == svc or cmdb_id.startswith(svc + '-') or cmdb_id.startswith(svc):
            return svc
    # fallback: return original cmdb_id (mapping failure noted)
    return cmdb_id

# apply mapping
cmdbs = cmdbs.copy()
cmdbs['service_name'] = cmdbs['cmdb_id'].apply(map_to_service)

# identify mapping failures (where service_name equals original cmdb_id and not a node)
mapping_failures = cmdbs[
    (cmdbs['service_name'] == cmdbs['cmdb_id']) & (~cmdbs['cmdb_id'].str.startswith('node-'))
]['cmdb_id'].drop_duplicates().reset_index(drop=True)

# 2) Aggregate per service_name
# Convert earliest_anomaly_ts back to epoch if possible: prefer numeric earliest_anom variable if present
if 'earliest_anom' in globals() and isinstance(earliest_anom, pd.DataFrame) and 'earliest_anomaly_ts' in earliest_anom.columns:
    # earliest_anom has columns cmdb_id, earliest_anomaly_ts (epoch)
    ea = earliest_anom.rename(columns={'earliest_anomaly_ts':'earliest_anom_epoch'})[['cmdb_id','earliest_anom_epoch']]
    cmdbs = cmdbs.merge(ea, on='cmdb_id', how='left')
else:
    # try to parse formatted strings in cmdbs['earliest_anomaly_ts'] back to epoch (if strings exist)
    def parse_epoch(x):
        if pd.isna(x) or x is None:
            return None
        try:
            dt = pd.to_datetime(x).tz_convert('Asia/Shanghai')
            return int(dt.tz_convert('UTC').timestamp())
        except Exception:
            return None
    cmdbs['earliest_anom_epoch'] = cmdbs['earliest_anomaly_ts'].apply(parse_epoch)

# Group by service_name
grp = cmdbs.groupby('service_name')

service_agg = grp.agg(
    service_total_points = ('total_points_in_window','sum'),
    service_anomalies_high = ('total_anomalies_high','sum'),
    service_anomalies_low = ('total_anomalies_low','sum'),
    pods_count = ('cmdb_id','nunique')
).reset_index()

service_agg['service_anomalies_total'] = service_agg['service_anomalies_high'] + service_agg['service_anomalies_low']

# pods_affected_count: number of pods of that service with anomalies_total > 0
pods_affected = cmdbs[cmdbs['anomalies_total'] > 0].groupby('service_name')['cmdb_id'].nunique().rename('pods_affected_count').reset_index()
service_agg = service_agg.merge(pods_affected, on='service_name', how='left')
service_agg['pods_affected_count'] = service_agg['pods_affected_count'].fillna(0).astype(int)

# earliest_service_anomaly_ts: min earliest_anom_epoch among pods (as localized string)
earliest_per_service = cmdbs.groupby('service_name')['earliest_anom_epoch'].min().rename('earliest_service_epoch').reset_index()
service_agg = service_agg.merge(earliest_per_service, on='service_name', how='left')

def fmt_epoch_nullable(x):
    if pd.isna(x) or x is None:
        return None
    return pd.to_datetime(int(x), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')

service_agg['earliest_service_anomaly_ts'] = service_agg['earliest_service_epoch'].apply(fmt_epoch_nullable)
service_agg = service_agg.drop(columns=['earliest_service_epoch'])

# top_5_pods_by_anomalies per service_name
top_pods = (
    cmdbs.sort_values(['service_name','anomalies_total'], ascending=[True, False])
         .groupby('service_name')
         .apply(lambda g: ", ".join(g.head(5)['cmdb_id'].tolist()))
         .rename('top_5_pods_by_anomalies')
         .reset_index()
)
service_agg = service_agg.merge(top_pods, on='service_name', how='left')

# 3) select and sort top 20 services by service_anomalies_total desc
service_summary = service_agg.sort_values(['service_anomalies_total','service_total_points'], ascending=[False, False]).head(20)

# Select requested columns and keep compact
service_summary = service_summary[[
    'service_name',
    'service_total_points',
    'service_anomalies_total',
    'service_anomalies_high',
    'service_anomalies_low',
    'pods_affected_count',
    'earliest_service_anomaly_ts',
    'top_5_pods_by_anomalies'
]].reset_index(drop=True)

# mapping_failures as a small dataframe for compact display
mapping_failures_df = mapping_failures.to_frame(name='unmapped_cmdb_id') if not mapping_failures.empty else pd.DataFrame(columns=['unmapped_cmdb_id'])

# Return compact results (service_summary top 20 and any mapping failures)
service_summary, mapping_failures_df

```
Out[6]:
```
Summary (service-level aggregation of metric anomalies for the incident window 2022-03-21 05:30–06:00 CST):

Overall
- Aggregated from per-pod (cmdb_id) results into service-level groups.
- Earliest anomalies across services cluster at ~2022-03-21 05:30:00 CST.
- Some cmdb_id did not map cleanly to a known service; these are noted below.

Top services by total anomalies (service_name: service_anomalies_total, service_total_points, pods_affected_count, earliest anomaly)
1. frontend: 668 anomalies, 17,608 points, 4 pods affected, earliest 2022-03-21 05:30:00 CST (top pods: frontend-2, frontend-1, frontend-0, frontend)
2. checkoutservice: 460 anomalies, 13,268 points, 4 pods affected, earliest 2022-03-21 05:30:00 CST (top pods: checkoutservice-1, checkoutservice-2, ...)
3. productcatalogservice: 400 anomalies, 11,160 points, 4 pods affected, earliest 2022-03-21 05:30:00 CST
4. adservice: 346 anomalies, 19,065 points, 4 pods affected, earliest 2022-03-21 05:30:00 CST
5. currencyservice: 298 anomalies, 7,812 points, 4 pods affected, earliest 2022-03-21 05:30:00 CST
6. cartservice: 277 anomalies, 9,300 points, 4 pods affected, earliest 2022-03-21 05:30:00 CST
7. recommendationservice: 226 anomalies, 9,238 points, 4 pods affected, earliest 05:30:00 CST
8. shippingservice: 212 anomalies, 7,564 points, 4 pods affected, earliest 05:30:00 CST
9. adservice2: 192 anomalies, 13,485 points, 2 pods affected, earliest 05:30:00 CST
10. paymentservice: 123 anomalies, 6,758 points, 4 pods affected, earliest 05:30:00 CST
(services 11–20 include emailservice, node-* entries, currencyservice2, cartservice2, frontend2, etc.; nodes show per-node anomalies as separate "service_name" rows)

Notes and interpretation
- Many core services (frontend, checkoutservice, productcatalogservice, adservice, currencyservice, cartservice) show the largest anomaly totals and multiple pods affected — this indicates broad, correlated deviations across services rather than isolated single-pod issues.
- Most services have anomalies across many KPIs (pods show dozens of distinct KPIs with anomalies).
- Nodes (node-2, node-3, node-4, node-5, node-6, node-1) also surface with non-trivial anomaly counts, suggesting potential infrastructure-level contributions.
- Earliest anomalies are concentrated at ~05:30 CST for top services, consistent with a system-wide perturbation starting around that time.

Mapping issues
- Some cmdb_id were left unmapped (kept under their original cmdb_id): examples include redis-cart-0 and redis-cart2-0 (and a few base-name entries). These unmapped cmdb_id are listed in the result and should be reviewed if you want them grouped under a service name.

If you want next steps, I can:
- expand the service table to include exact pod lists and counts for the top services,
- correlate these service-level anomalies with trace/log anomalies already computed, or
- rank services by anomalies per-point (anomaly density) to prioritize investigation.

The original code execution output of IPython Kernel is also provided below for reference:

(             service_name  service_total_points  service_anomalies_total  service_anomalies_high  service_anomalies_low  pods_affected_count earliest_service_anomaly_ts                            top_5_pods_by_anomalies
0                frontend                 17608                      668                     516                    152                    4     2022-03-21 05:30:00 CST       frontend-2, frontend-1, frontend-0, frontend
1         checkoutservice                 13268                      460                     288                    172                    4     2022-03-21 05:30:00 CST  checkoutservice-1, checkoutservice-2, checkout...
2   productcatalogservice                 11160                      400                     290                    110                    4     2022-03-21 05:30:00 CST  productcatalogservice-0, productcatalogservice...
3               adservice                 19065                      346                     282                     64                    4     2022-03-21 05:30:00 CST   adservice, adservice-0, adservice-2, adservice-1
4         currencyservice                  7812                      298                     210                     88                    4     2022-03-21 05:30:00 CST  currencyservice-2, currencyservice-0, currency...
5             cartservice                  9300                      277                     208                     69                    4     2022-03-21 05:30:00 CST  cartservice-0, cartservice-1, cartservice-2, c...
6   recommendationservice                  9238                      226                     171                     55                    4     2022-03-21 05:30:00 CST  recommendationservice-1, recommendationservice...
7         shippingservice                  7564                      212                     171                     41                    4     2022-03-21 05:30:00 CST  shippingservice-2, shippingservice-1, shipping...
8              adservice2                 13485                      192                     121                     71                    2     2022-03-21 05:30:00 CST                           adservice2, adservice2-0
9          paymentservice                  6758                      123                      79                     44                    4     2022-03-21 05:30:00 CST  paymentservice-0, paymentservice-1, paymentser...
10           emailservice                  7254                      101                      74                     27                    4     2022-03-21 05:30:00 CST  emailservice-0, emailservice-1, emailservice-2...
11                 node-5                  1748                       95                      60                     35                    1     2022-03-21 05:30:00 CST                                             node-5
12                 node-3                  1748                       93                      58                     35                    1     2022-03-21 05:30:00 CST                                             node-3
13                 node-2                  1779                       87                      53                     34                    1     2022-03-21 05:32:00 CST                                             node-2
14                 node-4                  1707                       83                      60                     23                    1     2022-03-21 05:30:00 CST                                             node-4
15                 node-6                  1781                       77                      31                     46                    1     2022-03-21 05:32:00 CST                                             node-6
16                 node-1                  1779                       61                      28                     33                    1     2022-03-21 05:31:00 CST                                             node-1
17       currencyservice2                  2604                       60                      25                     35                    1     2022-03-21 05:31:00 CST                                 currencyservice2-0
18           cartservice2                  3100                       47                      30                     17                    1     2022-03-21 05:30:00 CST                                     cartservice2-0
19              frontend2                  6172                       44                      29                     15                    1     2022-03-21 05:31:00 CST                                        frontend2-0,          unmapped_cmdb_id
0               adservice
1              adservice2
2             cartservice
3         checkoutservice
4         currencyservice
5            emailservice
6                frontend
7          paymentservice
8   productcatalogservice
9   recommendationservice
10           redis-cart-0
11          redis-cart2-0
12        shippingservice)```
```

In [ ]:
# In[7]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')
start_window = 1647811800
end_window = 1647813600

# Reuse df_metric from kernel
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')

# 1) Select mesh rows (mesh.source or mesh.destination) from full metric series
mesh_mask = df_metric['kpi_name'].str.contains('mesh.source', na=False) | df_metric['kpi_name'].str.contains('mesh.destination', na=False)
df_mesh_full = df_metric[mesh_mask].copy()

# If no mesh metrics, prepare fallback later
if df_mesh_full.empty:
    # compute anomalies_total per kpi_name across entire metric file (as fallback info)
    p95_all = df_metric.groupby(['cmdb_id','kpi_name'])['value'].quantile(0.95).rename('global_P95')
    p5_all = df_metric.groupby(['cmdb_id','kpi_name'])['value'].quantile(0.05).rename('global_P5')
    thresholds_all = pd.concat([p95_all, p5_all], axis=1).reset_index()
    window_all = df_metric[(df_metric['timestamp'] >= start_window) & (df_metric['timestamp'] <= end_window)].merge(thresholds_all, on=['cmdb_id','kpi_name'], how='left')
    window_all['is_high'] = (window_all['value'] > window_all['global_P95']).astype(int)
    window_all['is_low'] = (window_all['value'] < window_all['global_P5']).astype(int)
    agg_all = window_all.groupby(['cmdb_id','kpi_name']).agg(
        total_points_in_window=('value','size'),
        anomalies_high=('is_high','sum'),
        anomalies_low=('is_low','sum')
    ).reset_index()
    agg_all['anomalies_total'] = agg_all['anomalies_high'] + agg_all['anomalies_low']
    top20_fallback = agg_all.sort_values(['anomalies_total','total_points_in_window'], ascending=[False,False]).head(20).reset_index(drop=True)
    # Keep compact columns
    fallback = top20_fallback[['cmdb_id','kpi_name','total_points_in_window','anomalies_total','anomalies_high','anomalies_low']]
    fallback_message = "No mesh metrics found; returning top 20 metric (cmdb_id,kpi_name) by anomalies_total in window."
    fallback_message, fallback
else:
    # 2) For each distinct (cmdb_id,kpi_name) compute global P95 and P5 using full series (before window)
    p95_mesh = df_mesh_full.groupby(['cmdb_id','kpi_name'])['value'].quantile(0.95).rename('global_P95')
    p5_mesh = df_mesh_full.groupby(['cmdb_id','kpi_name'])['value'].quantile(0.05).rename('global_P5')
    thresholds_mesh = pd.concat([p95_mesh, p5_mesh], axis=1).reset_index()

    # 3) Filter mesh rows to incident window
    df_mesh_window = df_mesh_full[(df_mesh_full['timestamp'] >= start_window) & (df_mesh_full['timestamp'] <= end_window)].copy()

    # 4) Merge thresholds and compute anomalies per (cmdb_id,kpi_name)
    df_mesh_window = df_mesh_window.merge(thresholds_mesh, on=['cmdb_id','kpi_name'], how='left')
    df_mesh_window['is_high'] = (df_mesh_window['value'] > df_mesh_window['global_P95']).astype(int)
    df_mesh_window['is_low'] = (df_mesh_window['value'] < df_mesh_window['global_P5']).astype(int)

    grp = df_mesh_window.groupby(['cmdb_id','kpi_name'])
    agg = grp.agg(
        total_points_in_window=('value','size'),
        anomalies_high=('is_high','sum'),
        anomalies_low=('is_low','sum'),
        max_value_in_window=('value','max'),
        min_value_in_window=('value','min'),
        global_P95=('global_P95','first'),
        global_P5=('global_P5','first')
    ).reset_index()
    agg['anomalies_total'] = agg['anomalies_high'] + agg['anomalies_low']

    # earliest anomaly ts (high or low)
    anom_rows = df_mesh_window[(df_mesh_window['is_high']==1) | (df_mesh_window['is_low']==1)].copy()
    if not anom_rows.empty:
        earliest = anom_rows.groupby(['cmdb_id','kpi_name'])['timestamp'].min().rename('earliest_anomaly_ts').reset_index()
        agg = agg.merge(earliest, on=['cmdb_id','kpi_name'], how='left')
    else:
        agg['earliest_anomaly_ts'] = pd.NA

    # format earliest_anomaly_ts to Asia/Shanghai localized string or None
    def fmt_ts_nullable(x):
        if pd.isna(x):
            return None
        return pd.to_datetime(int(x), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
    agg['earliest_anomaly_ts'] = agg['earliest_anomaly_ts'].apply(fmt_ts_nullable)

    # 5) Parse kpi_name to extract source_service and dest_service
    def parse_mesh_kpi(kpi):
        toks = kpi.split('.')
        source = None
        dest = None
        if 'source' in toks:
            i = toks.index('source')
            if i+1 < len(toks):
                source = toks[i+1]
            if i+2 < len(toks):
                dest = toks[i+2]
        elif 'destination' in toks:
            i = toks.index('destination')
            if i+1 < len(toks):
                dest = toks[i+1]
            if i+2 < len(toks):
                source = toks[i+2]
        # if neither or incomplete, return (None,None)
        return source, dest

    parsed = agg['kpi_name'].apply(parse_mesh_kpi)
    agg['source_service'] = parsed.apply(lambda x: x[0])
    agg['dest_service'] = parsed.apply(lambda x: x[1])

    # For rows where parsing failed (both None), keep cmdb_id and kpi_name as-is by populating source_service with cmdb_id
    mask_unparsed = agg['source_service'].isna() & agg['dest_service'].isna()
    agg.loc[mask_unparsed, 'source_service'] = agg.loc[mask_unparsed, 'cmdb_id']
    agg.loc[mask_unparsed, 'dest_service'] = agg.loc[mask_unparsed, 'kpi_name']

    # 6) Return top 20 service-pair rows sorted by anomalies_total desc
    top20 = agg.sort_values(['anomalies_total','total_points_in_window'], ascending=[False,False]).head(20).reset_index(drop=True)

    # Select and order requested columns
    top20_result = top20[[
        'source_service','dest_service','cmdb_id','kpi_name',
        'total_points_in_window','anomalies_total','anomalies_high','anomalies_low',
        'earliest_anomaly_ts','max_value_in_window','min_value_in_window','global_P95','global_P5'
    ]]

    # Compact output
    top20_result

```
Out[7]:
```
The scan produced no visible output (the kernel returned None). Likely cause: the code path that handles the "no mesh metrics" case returned a fallback tuple but the final variable displayed (top20_result) was only set in the mesh-present branch, so nothing was returned.

What this means:
- The run did not present the requested top-20 mesh service-pair anomaly table.
- It’s unclear from this result whether there are zero mesh metrics in metric.csv or the script simply failed to return the fallback results.

Next steps I can take for you (pick one):
1. Rerun a corrected scan that always returns either: (a) top-20 mesh service-pairs with anomalies (if mesh metrics exist) or (b) a fallback top-20 non-mesh metric pairs by anomalies (if no mesh metrics exist).  
2. Immediately run a quick check to report whether any kpi_name contains 'mesh.source' or 'mesh.destination' and give a small sample (up to 20) so you can confirm presence/absence of mesh metrics.

Which would you like me to do?

The original code execution output of IPython Kernel is also provided below for reference:

None```
```

In [ ]:
# In[8]:

import pandas as pd
import pytz

# reuse existing df_metric if present
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')

tz = pytz.timezone('Asia/Shanghai')

# 1) select mesh rows
mesh_mask = df_metric['kpi_name'].str.contains('mesh.source|mesh.destination', na=False)
df_mesh = df_metric[mesh_mask].copy()

# 2) totals and distinct pairs
total_mesh_rows = int(df_mesh.shape[0])
distinct_mesh_pairs = int(df_mesh[['cmdb_id','kpi_name']].drop_duplicates().shape[0])

# 3) time range for mesh rows
if df_mesh.empty:
    time_summary = pd.DataFrame([{
        'min_timestamp_epoch': None,
        'max_timestamp_epoch': None,
        'min_timestamp_local': None,
        'max_timestamp_local': None
    }])
else:
    min_epoch = int(df_mesh['timestamp'].min())
    max_epoch = int(df_mesh['timestamp'].max())
    min_local = pd.to_datetime(min_epoch, unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
    max_local = pd.to_datetime(max_epoch, unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
    time_summary = pd.DataFrame([{
        'min_timestamp_epoch': min_epoch,
        'max_timestamp_epoch': max_epoch,
        'min_timestamp_local': min_local,
        'max_timestamp_local': max_local
    }])

# 4) top 20 most frequent (cmdb_id, kpi_name) pairs with counts
if df_mesh.empty:
    top20_pairs = pd.DataFrame(columns=['cmdb_id','kpi_name','count'])
else:
    top20_pairs = (
        df_mesh.groupby(['cmdb_id','kpi_name'])
        .size()
        .reset_index(name='count')
        .sort_values('count', ascending=False)
        .head(20)
        .reset_index(drop=True)
    )

# 5) up to 20 distinct kpi_name samples
if df_mesh.empty:
    sample_kpis = pd.DataFrame(columns=['kpi_name'])
else:
    sample_kpis = df_mesh['kpi_name'].drop_duplicates().head(20).to_frame().reset_index(drop=True)

# Compact outputs for display
total_mesh_rows, distinct_mesh_pairs, time_summary, top20_pairs, sample_kpis

```
Out[8]:
```
Summary of mesh-related metrics in metric.csv

- total_mesh_rows: 100,804 rows contain "mesh.source" or "mesh.destination".
- distinct_mesh_pairs (cmdb_id, kpi_name): 1,660 unique pairs.
- Time range for mesh rows:
  - epoch: min = 1647810000, max = 1647813600
  - Asia/Shanghai local: 2022-03-21 05:00:00 CST — 2022-03-21 06:00:00 CST

- Most frequent (cmdb_id, kpi_name) pairs (top examples, counts ≈ 61 each):
  - shippingservice2-0 — mesh.source.shippingservice2.jaeger-collector... (61)
  - adservice-0 — mesh.destination.frontend.adservice.istio_requests... (61)
  - adservice-0 — mesh.destination.frontend.adservice.istio_responses... (61)
  - adservice-0 — mesh.destination.unknown.adservice.istio_requests... (61)
  - (Many other adservice-related mesh destination/source KPIs appear at similar frequency.)

- Up to 20 distinct kpi_name samples observed (examples):
  - mesh.destination.frontend.adservice.istio_requests
  - mesh.destination.frontend.adservice.istio_responses
  - mesh.destination.unknown.adservice.istio_requests
  - mesh.source.shippingservice2.jaeger-collector... (jaeger-related mesh.source)
  - mesh.source.<service>.<service2>.istio_requests / istio_responses (many variants)
  - mesh.destination.<service>.<service2>.istio_* (multiple service pairs)

Interpretation: mesh (service-to-service) telemetry is abundant in the dataset (100k rows) and covers many service-pair KPIs (1.6k distinct pairs). The top-frequency entries are dominated by adservice-related destination metrics and some shippingservice2 source metrics. These mesh metrics are suitable for identifying anomalous inter-service call pairs in the incident window; if you want, I can now compute per-pair anomaly counts in the incident window and return the top 20 anomalous service-to-service pairs.

The original code execution output of IPython Kernel is also provided below for reference:

(100804, 1660,    min_timestamp_epoch  max_timestamp_epoch      min_timestamp_local      max_timestamp_local
0           1647810000           1647813600  2022-03-21 05:00:00 CST  2022-03-21 06:00:00 CST,                cmdb_id                                           kpi_name  count
0   shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....     61
1          adservice-0  mesh.destination.frontend.adservice.istio_requ...     61
2          adservice-0  mesh.destination.frontend.adservice.istio_requ...     61
3          adservice-0  mesh.destination.frontend.adservice.istio_requ...     61
4          adservice-0  mesh.destination.frontend.adservice.istio_requ...     61
5          adservice-0  mesh.destination.frontend.adservice.istio_requ...     61
6          adservice-0  mesh.destination.frontend.adservice.istio_requ...     61
7          adservice-0  mesh.destination.frontend.adservice.istio_requ...     61
8          adservice-0  mesh.destination.frontend.adservice.istio_requ...     61
9          adservice-0  mesh.destination.frontend.adservice.istio_requ...     61
10         adservice-0  mesh.destination.frontend.adservice.istio_requ...     61
11         adservice-0  mesh.destination.frontend.adservice.istio_resp...     61
12         adservice-0  mesh.destination.frontend.adservice.istio_resp...     61
13         adservice-0  mesh.destination.frontend.adservice.istio_resp...     61
14         adservice-0  mesh.destination.frontend.adservice.istio_resp...     61
15         adservice-0  mesh.destination.unknown.adservice.istio_reque...     61
16         adservice-0  mesh.destination.unknown.adservice.istio_reque...     61
17         adservice-0  mesh.destination.unknown.adservice.istio_reque...     61
18         adservice-0  mesh.destination.unknown.adservice.istio_reque...     61
19         adservice-0  mesh.destination.unknown.adservice.istio_reque...     61,                                              kpi_name
0   mesh.destination.frontend.adservice.istio_requ...
1   mesh.destination.frontend.adservice.istio_requ...
2   mesh.destination.frontend.adservice.istio_requ...
3   mesh.destination.frontend.adservice.istio_requ...
4   mesh.destination.frontend.adservice.istio_requ...
5   mesh.destination.frontend.adservice.istio_requ...
6   mesh.destination.frontend.adservice.istio_requ...
7   mesh.destination.frontend.adservice.istio_requ...
8   mesh.destination.frontend.adservice.istio_requ...
9   mesh.destination.frontend.adservice.istio_requ...
10  mesh.destination.frontend.adservice.istio_resp...
11  mesh.destination.frontend.adservice.istio_resp...
12  mesh.destination.frontend.adservice.istio_resp...
13  mesh.destination.frontend.adservice.istio_resp...
14  mesh.destination.unknown.adservice.istio_reque...
15  mesh.destination.unknown.adservice.istio_reque...
16  mesh.destination.unknown.adservice.istio_reque...
17  mesh.destination.unknown.adservice.istio_reque...
18  mesh.destination.unknown.adservice.istio_reque...
19  mesh.destination.unknown.adservice.istio_reque...)```
```

In [ ]:
# In[9]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')
start_window = 1647811800
end_window = 1647813600

# reuse df_metric if present
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')

# 1) select mesh rows (mesh.source or mesh.destination)
mesh_mask = df_metric['kpi_name'].str.contains('mesh.source|mesh.destination', na=False)
df_mesh_full = df_metric[mesh_mask].copy()

# Prepare empty outputs default
top20_kpis = pd.DataFrame(columns=[
    'source_service','dest_service','cmdb_id','kpi_name',
    'total_points_in_window','anomalies_total','anomalies_high','anomalies_low',
    'earliest_anomaly_ts','global_P95','global_P5'
])
top20_pairs = pd.DataFrame(columns=[
    'source_service','dest_service','pair_anomalies_total','pair_total_points',
    'pair_earliest_anomaly_ts','number_of_distinct_kpis'
])

if not df_mesh_full.empty:
    # 2) global thresholds per (cmdb_id,kpi_name) using full mesh series
    p95 = df_mesh_full.groupby(['cmdb_id','kpi_name'])['value'].quantile(0.95).rename('global_P95')
    p5 = df_mesh_full.groupby(['cmdb_id','kpi_name'])['value'].quantile(0.05).rename('global_P5')
    thresholds = pd.concat([p95, p5], axis=1).reset_index()

    # 3) filter to incident window
    df_mesh_window = df_mesh_full[(df_mesh_full['timestamp'] >= start_window) & (df_mesh_full['timestamp'] <= end_window)].copy()

    if not df_mesh_window.empty:
        # 4) merge thresholds and compute per-(cmdb_id,kpi_name) stats
        df_mesh_window = df_mesh_window.merge(thresholds, on=['cmdb_id','kpi_name'], how='left')
        df_mesh_window['is_high'] = (df_mesh_window['value'] > df_mesh_window['global_P95']).astype(int)
        df_mesh_window['is_low'] = (df_mesh_window['value'] < df_mesh_window['global_P5']).astype(int)

        grp = df_mesh_window.groupby(['cmdb_id','kpi_name'])
        agg = grp.agg(
            total_points_in_window = ('value','size'),
            anomalies_high = ('is_high','sum'),
            anomalies_low = ('is_low','sum'),
            max_value_in_window = ('value','max'),
            min_value_in_window = ('value','min'),
            global_P95 = ('global_P95','first'),
            global_P5 = ('global_P5','first')
        ).reset_index()
        agg['anomalies_total'] = agg['anomalies_high'] + agg['anomalies_low']

        # earliest anomaly ts per (cmdb_id,kpi_name)
        anom_rows = df_mesh_window[(df_mesh_window['is_high']==1) | (df_mesh_window['is_low']==1)].copy()
        if not anom_rows.empty:
            earliest = anom_rows.groupby(['cmdb_id','kpi_name'])['timestamp'].min().rename('earliest_anom_epoch').reset_index()
            agg = agg.merge(earliest, on=['cmdb_id','kpi_name'], how='left')
        else:
            agg['earliest_anom_epoch'] = pd.NA

        # 5) parse kpi_name to extract source and dest services
        def parse_pair(kpi):
            toks = kpi.split('.')
            source = None
            dest = None
            if 'source' in toks:
                i = toks.index('source')
                if i+1 < len(toks):
                    source = toks[i+1]
                if i+2 < len(toks):
                    dest = toks[i+2]
            elif 'destination' in toks:
                i = toks.index('destination')
                if i+1 < len(toks):
                    source = toks[i+1]
                if i+2 < len(toks):
                    dest = toks[i+2]
            return source, dest

        parsed = agg['kpi_name'].apply(parse_pair)
        agg['source_service'] = parsed.apply(lambda x: x[0])
        agg['dest_service'] = parsed.apply(lambda x: x[1])

        # format earliest_anom_epoch to localized string
        def fmt_epoch(x):
            if pd.isna(x) or x is None:
                return None
            return pd.to_datetime(int(x), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
        agg['earliest_anomaly_ts'] = agg['earliest_anom_epoch'].apply(fmt_epoch)
        agg = agg.drop(columns=['earliest_anom_epoch'])

        # 6) Aggregate by service-pair
        service_group = agg.groupby(['source_service','dest_service']).agg(
            pair_total_points = ('total_points_in_window','sum'),
            pair_anomalies_total = ('anomalies_total','sum'),
            pair_anomalies_high = ('anomalies_high','sum'),
            pair_anomalies_low = ('anomalies_low','sum'),
            number_of_distinct_kpis = ('kpi_name','nunique')
        ).reset_index()

        # compute earliest anomaly per pair (min of per-kpi earliest where not None)
        def parse_epoch_from_str(s):
            if s is None or (isinstance(s,float) and pd.isna(s)):
                return None
            try:
                dt = pd.to_datetime(s)
                if dt.tzinfo is None:
                    dt = dt.tz_localize(tz)
                return int(dt.tz_convert('UTC').timestamp())
            except Exception:
                return None

        agg['earliest_epoch_for_min'] = agg['earliest_anomaly_ts'].apply(parse_epoch_from_str)
        earliest_pair = agg.groupby(['source_service','dest_service'])['earliest_epoch_for_min'].min().rename('pair_earliest_epoch').reset_index()
        service_group = service_group.merge(earliest_pair, on=['source_service','dest_service'], how='left')
        service_group['pair_earliest_anomaly_ts'] = service_group['pair_earliest_epoch'].apply(lambda x: fmt_epoch(x) if pd.notna(x) else None)
        service_group = service_group.drop(columns=['pair_earliest_epoch'])

        # 7a) Top 20 individual mesh kpi rows by anomalies_total desc
        top20_kpis = agg.sort_values(['anomalies_total','total_points_in_window'], ascending=[False,False]).head(20).copy()
        top20_kpis['kpi_name'] = top20_kpis['kpi_name'].apply(lambda s: s if len(s)<=80 else s[:77]+'...')
        top20_kpis = top20_kpis[[
            'source_service','dest_service','cmdb_id','kpi_name',
            'total_points_in_window','anomalies_total','anomalies_high','anomalies_low',
            'earliest_anomaly_ts','global_P95','global_P5'
        ]].reset_index(drop=True)

        # 7b) Top 20 aggregated service-pairs by pair_anomalies_total desc
        top20_pairs = service_group.sort_values(['pair_anomalies_total','pair_total_points'], ascending=[False,False]).head(20).reset_index(drop=True)
        top20_pairs = top20_pairs[[
            'source_service','dest_service','pair_anomalies_total','pair_total_points',
            'pair_earliest_anomaly_ts','number_of_distinct_kpis'
        ]]

# Final compact outputs
top20_kpis, top20_pairs

```
Out[9]:
```
Summary of mesh (service-to-service) anomaly scan for the incident window (2022-03-21 05:30–06:00 CST):

Key facts
- We analyzed mesh metrics (kpi_name containing "mesh.source" or "mesh.destination") in the incident window and computed global P95/P5 per (cmdb_id, kpi_name) using the full series.
- Individual mesh KPI rows: many show anomalies during the window (top rows each had anomalies_total = 6 across 31 samples).
- Earliest anomalies for top items occurred around 2022-03-21 05:30:00 CST — 05:33:00 CST.

Top individual mesh KPI findings (examples from the top 20)
- Several checkoutservice-1 mesh KPIs are anomalous (each: 31 points, anomalies_total = 6, earliest anomaly 05:30:00 CST), e.g.:
  - mesh.destination.frontend.checkoutservice.... (large request/response counts, global_P95 ~1570 / 2900 etc.)
  - mesh.source.checkoutservice.cartservice.... (high counts with global_P95 values like 6400, 760, etc.)
  - mesh.source.checkoutservice.currencyservice..., emailservice..., jaeger-collector..., paymentservice..., productcatalogservice... — all with anomalies_total = 6 and earliest 05:30:00 CST.
- One adservice2-related KPI from cmdb_id adservice2-0 (mesh.destination.unknown.adservice2...) shows anomalies_total = 6 (earliest 05:33:00 CST).
- In general each top mesh KPI had 31 points in-window and 3 high + 3 low anomalies relative to global thresholds.

Top aggregated service-pair results (service-level, summed over all mesh KPIs for a pair)
- Highest aggregated anomaly totals (pair_anomalies_total):
  1. frontend -> productcatalogservice : 144 anomalies (pair_total_points = 2,976), earliest 05:32:00 CST, 32 distinct KPIs
  2. frontend -> currencyservice       : 141 anomalies (1,984 pts), earliest 05:33:00 CST, 24 KPIs
  3. frontend -> cartservice           : 133 anomalies (2,852 pts), earliest 05:32:00 CST, 32 KPIs
  4. frontend -> recommendationservice : 130 anomalies (2,604 pts), earliest 05:33:00 CST, 28 KPIs
  5. recommendationservice -> productcatalogservice : 126 anomalies (1,488 pts), earliest 05:32:00 CST, 16 KPIs
  6. frontend -> adservice             : 114 anomalies (2,604 pts), earliest 05:32:00 CST, 28 KPIs
  7. frontend -> shippingservice       : 108 anomalies (1,116 pts), earliest 05:30:00 CST, 12 KPIs
  8. checkoutservice -> currencyservice: 100 anomalies (1,240 pts), earliest 05:30:00 CST, 16 KPIs
  9. checkoutservice -> productcatalogservice: 96 anomalies (2,480 pts), earliest 05:30:00 CST, 28 KPIs
 10. checkoutservice -> cartservice    : 84 anomalies (2,356 pts), earliest 05:30:00 CST, 28 KPIs
 11. frontend -> checkoutservice      : 82 anomalies (2,604 pts), earliest 05:30:00 CST, 28 KPIs
 12. checkoutservice -> paymentservice: 82 anomalies (1,240 pts), earliest 05:30:00 CST, 16 KPIs
- Additional flagged pairs include unknown->frontend, productcatalogservice->jaeger-collector, frontend->jaeger-collector, adservice->jaeger-collector, shippingservice->jaeger-collector, etc.

Interpretation / guidance
- The largest aggregated anomalies involve frontend as a source calling many downstream services (productcatalogservice, currencyservice, cartservice, recommendationservice, adservice, checkoutservice, shippingservice). That pattern indicates widespread deviations in inter-service traffic originating from frontend during the incident window.
- checkoutservice is also a major anomalous source, impacting many downstream services (currency, productcatalog, cart, payment, email, jaeger).
- Individual mesh KPIs typically show small per-KPI anomaly counts (e.g., 6 of 31 samples), but aggregated across many KPIs and pods the pair-level anomaly totals become large and help prioritize investigation.
- Earliest anomalies cluster at ~05:30–05:33 CST, consistent with the incident start identified earlier.

If you want, next steps I can run:
- Show the full (truncated) top-20 tables with exact numeric fields for quick inspection, or
- Correlate the top service-pair anomalies with node/pod metric anomalies and trace/log anomalies to find likely root causes. Which would you like?

The original code execution output of IPython Kernel is also provided below for reference:

(     source_service           dest_service            cmdb_id                                           kpi_name  total_points_in_window  anomalies_total  anomalies_high  anomalies_low      earliest_anomaly_ts    global_P95    global_P5
0           unknown             adservice2       adservice2-0  mesh.destination.unknown.adservice2.istio_requ...                      31                6               3              3  2022-03-21 05:33:00 CST      0.138700     0.128200
1          frontend        checkoutservice  checkoutservice-1  mesh.destination.frontend.checkoutservice.isti...                      31                6               3              3  2022-03-21 05:30:00 CST   1570.000000   392.500000
2          frontend        checkoutservice  checkoutservice-1  mesh.destination.frontend.checkoutservice.isti...                      31                6               3              3  2022-03-21 05:30:00 CST   2900.000000   725.000000
3   checkoutservice            cartservice  checkoutservice-1  mesh.source.checkoutservice.cartservice.istio_...                      31                6               3              3  2022-03-21 05:30:00 CST   6400.000000  1600.000000
4   checkoutservice            cartservice  checkoutservice-1  mesh.source.checkoutservice.cartservice.istio_...                      31                6               3              3  2022-03-21 05:30:00 CST      0.009200     0.001550
5   checkoutservice            cartservice  checkoutservice-1  mesh.source.checkoutservice.cartservice.istio_...                      31                6               3              3  2022-03-21 05:30:00 CST    760.000000   190.000000
6   checkoutservice        currencyservice  checkoutservice-1  mesh.source.checkoutservice.currencyservice.is...                      31                6               3              3  2022-03-21 05:30:00 CST   9300.000000  2325.000000
7   checkoutservice        currencyservice  checkoutservice-1  mesh.source.checkoutservice.currencyservice.is...                      31                6               3              3  2022-03-21 05:30:00 CST      0.008375     0.001575
8   checkoutservice        currencyservice  checkoutservice-1  mesh.source.checkoutservice.currencyservice.is...                      31                6               3              3  2022-03-21 05:30:00 CST   1350.000000   337.500000
9   checkoutservice           emailservice  checkoutservice-1  mesh.source.checkoutservice.emailservice.istio...                      31                6               3              3  2022-03-21 05:30:00 CST   3700.000000   925.000000
10  checkoutservice           emailservice  checkoutservice-1  mesh.source.checkoutservice.emailservice.istio...                      31                6               3              3  2022-03-21 05:30:00 CST      0.004100     0.001025
11  checkoutservice           emailservice  checkoutservice-1  mesh.source.checkoutservice.emailservice.istio...                      31                6               3              3  2022-03-21 05:30:00 CST    430.000000   107.500000
12  checkoutservice       jaeger-collector  checkoutservice-1  mesh.source.checkoutservice.jaeger-collector.i...                      31                6               3              3  2022-03-21 05:30:00 CST  28100.000000  7025.000000
13  checkoutservice       jaeger-collector  checkoutservice-1  mesh.source.checkoutservice.jaeger-collector.i...                      31                6               3              3  2022-03-21 05:30:00 CST      4.000000     1.000000
14  checkoutservice       jaeger-collector  checkoutservice-1  mesh.source.checkoutservice.jaeger-collector.i...                      31                6               3              3  2022-03-21 05:30:00 CST    398.000000    99.500000
15  checkoutservice         paymentservice  checkoutservice-1  mesh.source.checkoutservice.paymentservice.ist...                      31                6               3              3  2022-03-21 05:30:00 CST   3300.000000   825.000000
16  checkoutservice         paymentservice  checkoutservice-1  mesh.source.checkoutservice.paymentservice.ist...                      31                6               3              3  2022-03-21 05:30:00 CST      0.003600     0.000525
17  checkoutservice         paymentservice  checkoutservice-1  mesh.source.checkoutservice.paymentservice.ist...                      31                6               3              3  2022-03-21 05:30:00 CST    510.000000   127.500000
18  checkoutservice  productcatalogservice  checkoutservice-1  mesh.source.checkoutservice.productcatalogserv...                      31                6               3              3  2022-03-21 05:30:00 CST   6600.000000  1650.000000
19  checkoutservice  productcatalogservice  checkoutservice-1  mesh.source.checkoutservice.productcatalogserv...                      31                6               3              3  2022-03-21 05:30:00 CST   1210.000000   285.000000,            source_service           dest_service  pair_anomalies_total  pair_total_points pair_earliest_anomaly_ts  number_of_distinct_kpis
0                frontend  productcatalogservice                   144               2976  2022-03-21 05:32:00 CST                       32
1                frontend        currencyservice                   141               1984  2022-03-21 05:33:00 CST                       24
2                frontend            cartservice                   133               2852  2022-03-21 05:32:00 CST                       32
3                frontend  recommendationservice                   130               2604  2022-03-21 05:33:00 CST                       28
4   recommendationservice  productcatalogservice                   126               1488  2022-03-21 05:32:00 CST                       16
5                frontend              adservice                   114               2604  2022-03-21 05:32:00 CST                       28
6                frontend        shippingservice                   108               1116  2022-03-21 05:30:00 CST                       12
7         checkoutservice        currencyservice                   100               1240  2022-03-21 05:30:00 CST                       16
8         checkoutservice  productcatalogservice                    96               2480  2022-03-21 05:30:00 CST                       28
9         checkoutservice            cartservice                    84               2356  2022-03-21 05:30:00 CST                       28
10        checkoutservice        shippingservice                    84               1116  2022-03-21 05:30:00 CST                       12
11               frontend        checkoutservice                    82               2604  2022-03-21 05:30:00 CST                       28
12        checkoutservice         paymentservice                    82               1240  2022-03-21 05:30:00 CST                       16
13                unknown               frontend                    79               1488  2022-03-21 05:33:00 CST                       16
14        checkoutservice           emailservice                    54               1736  2022-03-21 05:30:00 CST                       20
15  productcatalogservice             basic-tidb                    52                744  2022-03-21 05:32:00 CST                        8
16  productcatalogservice       jaeger-collector                    49                744  2022-03-21 05:32:00 CST                        8
17               frontend       jaeger-collector                    45                372  2022-03-21 05:33:00 CST                        4
18              adservice       jaeger-collector                    36                620  2022-03-21 05:32:00 CST                       12
19        shippingservice       jaeger-collector                    32                372  2022-03-21 05:33:00 CST                        4)```
```

In [ ]:
# In[10]:

import pandas as pd
import pytz

# reuse existing df_trace if available
try:
    df_trace
except NameError:
    df_trace = pd.read_csv('trace.csv')

tz = pytz.timezone('Asia/Shanghai')
start_window = 1647811800
end_window = 1647813600

# 1) Global thresholds per (cmdb_id, trace_name) using full series (do NOT filter first)
if df_trace.empty:
    thresholds_trace = pd.DataFrame(columns=['cmdb_id','trace_name','global_P95','global_P5'])
else:
    p95 = df_trace.groupby(['cmdb_id','trace_name'])['value'].quantile(0.95).rename('global_P95')
    p5 = df_trace.groupby(['cmdb_id','trace_name'])['value'].quantile(0.05).rename('global_P5')
    thresholds_trace = pd.concat([p95, p5], axis=1).reset_index()

# 2) Filter to incident window
window_trace = df_trace[(df_trace['timestamp'] >= start_window) & (df_trace['timestamp'] <= end_window)].copy()

# If no window rows, prepare empty outputs
if window_trace.empty:
    distinct_cmdb_count = 0
    cmdbs_with_anomalies_count = 0
    top20_summary = pd.DataFrame(columns=[
        'cmdb_id','total_trace_points_in_window','total_trace_anomalies',
        'total_trace_anomalies_high','total_trace_anomalies_low',
        'distinct_trace_names_with_anomalies','earliest_trace_anomaly_ts',
        'top_5_trace_names_by_anomaly_count'
    ])
else:
    # 3) Merge thresholds into window rows and compute per-row anomaly flags
    window_trace = window_trace.merge(thresholds_trace, on=['cmdb_id','trace_name'], how='left')
    window_trace['is_high'] = (window_trace['value'] > window_trace['global_P95']).astype(int)
    window_trace['is_low'] = (window_trace['value'] < window_trace['global_P5']).astype(int)
    window_trace['is_anom'] = ((window_trace['is_high'] + window_trace['is_low']) > 0).astype(int)

    # Aggregate per cmdb_id
    cmdb_agg = window_trace.groupby('cmdb_id').agg(
        total_trace_points_in_window = ('value','size'),
        total_trace_anomalies_high = ('is_high','sum'),
        total_trace_anomalies_low = ('is_low','sum')
    ).reset_index()
    cmdb_agg['total_trace_anomalies'] = cmdb_agg['total_trace_anomalies_high'] + cmdb_agg['total_trace_anomalies_low']

    # earliest_trace_anomaly_ts per cmdb_id
    anom_rows = window_trace.loc[window_trace['is_anom']==1].copy()
    if not anom_rows.empty:
        earliest = anom_rows.groupby('cmdb_id')['timestamp'].min().rename('earliest_epoch').reset_index()
        cmdb_agg = cmdb_agg.merge(earliest, on='cmdb_id', how='left')
    else:
        cmdb_agg['earliest_epoch'] = pd.NA

    # distinct_trace_names_with_anomalies and top_5_trace_names_by_anomaly_count
    per_trace_anom = window_trace.groupby(['cmdb_id','trace_name']).agg(
        anomalies_high = ('is_high','sum'),
        anomalies_low = ('is_low','sum')
    ).reset_index()
    per_trace_anom['anomalies_total'] = per_trace_anom['anomalies_high'] + per_trace_anom['anomalies_low']

    distinct_counts = (
        per_trace_anom[per_trace_anom['anomalies_total'] > 0]
        .groupby('cmdb_id')
        .size()
        .rename('distinct_trace_names_with_anomalies')
        .reset_index()
    )
    cmdb_agg = cmdb_agg.merge(distinct_counts, on='cmdb_id', how='left')
    cmdb_agg['distinct_trace_names_with_anomalies'] = cmdb_agg['distinct_trace_names_with_anomalies'].fillna(0).astype(int)

    # top 5 trace names by anomaly count per cmdb_id (format "name(count)")
    def top5_traces(subdf):
        s = subdf.sort_values('anomalies_total', ascending=False).head(5)
        items = s.apply(lambda r: f"{r['trace_name']}({int(r['anomalies_total'])})", axis=1).tolist()
        return ", ".join(items)

    top5 = per_trace_anom.groupby('cmdb_id').apply(lambda g: top5_traces(g)).rename('top_5_trace_names_by_anomaly_count').reset_index()
    cmdb_agg = cmdb_agg.merge(top5, on='cmdb_id', how='left')
    cmdb_agg['top_5_trace_names_by_anomaly_count'] = cmdb_agg['top_5_trace_names_by_anomaly_count'].fillna('')

    # format earliest timestamp to Asia/Shanghai
    def fmt_epoch_nullable(x):
        if pd.isna(x) or x is None:
            return None
        return pd.to_datetime(int(x), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')

    cmdb_agg['earliest_trace_anomaly_ts'] = cmdb_agg['earliest_epoch'].apply(fmt_epoch_nullable)
    cmdb_agg = cmdb_agg.drop(columns=['earliest_epoch'])

    # 4) Prepare final top 20 by total_trace_anomalies desc; if no anomalies anywhere, sort by points
    distinct_cmdb_count = cmdb_agg['cmdb_id'].nunique()
    cmdbs_with_anomalies_count = (cmdb_agg['total_trace_anomalies'] > 0).sum()

    if cmdb_agg['total_trace_anomalies'].sum() == 0:
        top20_summary = cmdb_agg.sort_values(['total_trace_points_in_window','cmdb_id'], ascending=[False,True]).head(20)
    else:
        top20_summary = cmdb_agg.sort_values(['total_trace_anomalies','total_trace_points_in_window','cmdb_id'], ascending=[False,False,True]).head(20)

    top20_summary = top20_summary[[
        'cmdb_id',
        'total_trace_points_in_window',
        'total_trace_anomalies',
        'total_trace_anomalies_high',
        'total_trace_anomalies_low',
        'distinct_trace_names_with_anomalies',
        'earliest_trace_anomaly_ts',
        'top_5_trace_names_by_anomaly_count'
    ]].reset_index(drop=True)

# Final compact outputs (variable names for IPython display)
distinct_cmdb_count, cmdbs_with_anomalies_count, top20_summary

```
Out[10]:
```
Summary of trace-level anomaly aggregation for the incident window (2022-03-21 05:30–06:00 CST / Asia/Shanghai):

Overview
- Distinct cmdb_id processed: 40  
- cmdb_id with trace anomalies: 40 (all shown had anomalies)
- Earliest anomalies appear around 2022-03-21 05:30:00 CST.

Top cmdb_id by trace anomalies (compact — top 20 truncated to top 10 for brevity)
1. frontend-0 — points=2624, anomalies=200 (high=108, low=92), distinct_trace_names_with_anomalies=65, earliest=2022-03-21 05:30:00 CST. Top traces: trace.from_root.duration_mean(...), trace.from_root.duration_p95(...), ...
2. frontend-1 — points=2628, anomalies=199 (high=102, low=97), distinct=64, earliest=05:30:00 CST.
3. frontend-2 — points=2432, anomalies=180 (high=100, low=80), distinct=61, earliest=05:30:00 CST.
4. checkoutservice-2 — points=1856, anomalies=121 (high=72, low=49), distinct=55, earliest=05:30:00 CST.
5. checkoutservice-0 — points=1864, anomalies=116 (high=70, low=46), distinct=50, earliest=05:31:00 CST.
6. checkoutservice-1 — points=1936, anomalies=115 (high=64, low=51), distinct=45, earliest=05:30:00 CST.
7. productcatalogservice-1 — points=1012, anomalies=89 (high=46, low=43), distinct=29, earliest=05:30:00 CST.
8. productcatalogservice-0 — points=1012, anomalies=83 (high=43, low=40), distinct=27, earliest=05:31:00 CST.
9. productcatalogservice-2 — points=1024, anomalies=81 (high=44, low=37), distinct=28, earliest=05:30:00 CST.
10. recommendationservice-1 — points=840, anomalies=73 (high=37, low=36), distinct=21, earliest=05:30:00 CST.

(Other top-20 entries include recommendationservice-0/2, currencyservice-0/1/2, shippingservice-*, frontend2-0, checkoutservice2-0, etc.; each row shows multiple trace metrics with anomalies.)

Interpretation (concise)
- Trace anomalies are widespread and concentrated in the same time window (~05:30–05:33 CST).  
- Frontend pods show the largest number of trace anomalies (top three cmdb_id), followed by checkoutservice and productcatalogservice pods.  
- Many services have dozens of distinct trace names affected, indicating broad, correlated latency/trace deviations rather than single-trace outliers.

Suggested next steps
- Correlate these top cmdb_id trace anomalies with the previously computed metric and mesh service-pair anomalies (frontend & checkoutservice were high on those lists as well).  
- Inspect the top trace_names (e.g., trace.from_root.*, trace.from_frontend.*, trace.self.*) for error rates or latency spikes around the earliest timestamps (~05:30) to narrow root cause.  
- If you want, I can produce the full top-20 table with exact trace-name strings and counts, or cross-correlate trace anomalies with mesh/service-pair anomalies. Which would you prefer?

The original code execution output of IPython Kernel is also provided below for reference:

(40, np.int64(40),                     cmdb_id  total_trace_points_in_window  total_trace_anomalies  total_trace_anomalies_high  total_trace_anomalies_low  distinct_trace_names_with_anomalies earliest_trace_anomaly_ts                 top_5_trace_names_by_anomaly_count
0                frontend-0                          2624                    200                         108                         92                                   65   2022-03-21 05:30:00 CST  trace.from_root.duration_mean(4), trace.from_r...
1                frontend-1                          2628                    199                         102                         97                                   64   2022-03-21 05:30:00 CST  trace.from_root.duration_mean(4), trace.from_r...
2                frontend-2                          2432                    180                         100                         80                                   61   2022-03-21 05:30:00 CST  trace.from_root.duration_mean(4), trace.from_r...
3         checkoutservice-2                          1856                    121                          72                         49                                   55   2022-03-21 05:30:00 CST  trace.self.duration_mean(4), trace.self.durati...
4         checkoutservice-0                          1864                    116                          70                         46                                   50   2022-03-21 05:31:00 CST  trace.self.duration_mean(4), trace.self.durati...
5         checkoutservice-1                          1936                    115                          64                         51                                   45   2022-03-21 05:30:00 CST  trace.self.duration_mean(4), trace.self.durati...
6   productcatalogservice-1                          1012                     89                          46                         43                                   29   2022-03-21 05:30:00 CST  trace.from_checkoutservice-1.duration_p95(4), ...
7   productcatalogservice-0                          1012                     83                          43                         40                                   27   2022-03-21 05:31:00 CST  trace.from_checkoutservice-0.duration_mean(4),...
8   productcatalogservice-2                          1024                     81                          44                         37                                   28   2022-03-21 05:30:00 CST  trace.from_checkoutservice-1.duration_p95(4), ...
9   recommendationservice-1                           840                     73                          37                         36                                   21   2022-03-21 05:30:00 CST  trace.from_frontend-0.duration_mean(4), trace....
10  recommendationservice-0                           840                     69                          37                         32                                   21   2022-03-21 05:30:00 CST  trace.from_frontend-0.duration_mean(4), trace....
11  recommendationservice-2                           840                     67                          34                         33                                   21   2022-03-21 05:31:00 CST  trace.from_frontend-0.duration_mean(4), trace....
12        currencyservice-0                           668                     56                          32                         24                                   17   2022-03-21 05:30:00 CST  trace.from_checkoutservice-0.duration_p95(4), ...
13        currencyservice-2                           668                     56                          30                         26                                   16   2022-03-21 05:30:00 CST  trace.from_checkoutservice-0.duration_mean(4),...
14        currencyservice-1                           660                     55                          28                         27                                   17   2022-03-21 05:30:00 CST  trace.from_checkoutservice-0.duration_mean(4),...
15        shippingservice-1                           600                     48                          27                         21                                   15   2022-03-21 05:30:00 CST  trace.from_frontend-1.duration_mean(4), trace....
16        shippingservice-2                           616                     44                          27                         17                                   16   2022-03-21 05:32:00 CST  trace.from_checkoutservice-1.duration_p95(4), ...
17              frontend2-0                           308                     43                          25                         18                                   25   2022-03-21 05:30:00 CST  trace.from_root.duration_mean(2), trace.from_r...
18        shippingservice-0                           620                     37                          23                         14                                   15   2022-03-21 05:31:00 CST  trace.from_checkoutservice-0.duration_p95(4), ...
19       checkoutservice2-0                           244                     34                          16                         18                                   20   2022-03-21 05:30:00 CST  trace.from_frontend2-0.duration_mean(2), trace...)```
```

In [ ]:
# In[11]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')
start_window = 1647811800
end_window = 1647813600

# reuse df_trace if present
try:
    df_trace
except NameError:
    df_trace = pd.read_csv('trace.csv')

# 1) Compute global thresholds per (cmdb_id, trace_name) using full series (if not already computed)
if 'thresholds_trace' not in globals():
    if df_trace.empty:
        thresholds_trace = pd.DataFrame(columns=['cmdb_id','trace_name','global_P95','global_P5'])
    else:
        p95 = df_trace.groupby(['cmdb_id','trace_name'])['value'].quantile(0.95).rename('global_P95')
        p5 = df_trace.groupby(['cmdb_id','trace_name'])['value'].quantile(0.05).rename('global_P5')
        thresholds_trace = pd.concat([p95, p5], axis=1).reset_index()

# 2) Filter trace rows for checkoutservice-1 within incident window
df_cs1_window = df_trace[
    (df_trace['cmdb_id'] == 'checkoutservice-1') &
    (df_trace['timestamp'] >= start_window) &
    (df_trace['timestamp'] <= end_window)
].copy()

# If no rows, prepare empty compact result
cols = [
    'trace_name','total_points_in_window','anomalies_high','earliest_high_ts',
    'anomalies_low','earliest_low_ts','anomalies_total',
    'max_value_in_window','min_value_in_window','global_P95','global_P5'
]
if df_cs1_window.empty:
    top20_traces = pd.DataFrame(columns=cols)
else:
    # 3) Merge global thresholds (computed from full series) into the window rows
    df_cs1_window = df_cs1_window.merge(
        thresholds_trace[thresholds_trace['cmdb_id']=='checkoutservice-1'],
        on=['cmdb_id','trace_name'],
        how='left'
    )
    # 4) flag anomalies per row
    df_cs1_window['is_high'] = (df_cs1_window['value'] > df_cs1_window['global_P95']).astype(int)
    df_cs1_window['is_low'] = (df_cs1_window['value'] < df_cs1_window['global_P5']).astype(int)

    # 5) aggregate per trace_name
    grp = df_cs1_window.groupby('trace_name')
    agg = grp.agg(
        total_points_in_window = ('value','size'),
        anomalies_high = ('is_high','sum'),
        anomalies_low = ('is_low','sum'),
        max_value_in_window = ('value','max'),
        min_value_in_window = ('value','min'),
        global_P95 = ('global_P95','first'),
        global_P5 = ('global_P5','first')
    ).reset_index()
    agg['anomalies_total'] = agg['anomalies_high'] + agg['anomalies_low']

    # earliest timestamps for high/low
    earliest_high = df_cs1_window.loc[df_cs1_window['is_high']==1].groupby('trace_name')['timestamp'].min().rename('earliest_high_epoch').reset_index()
    earliest_low = df_cs1_window.loc[df_cs1_window['is_low']==1].groupby('trace_name')['timestamp'].min().rename('earliest_low_epoch').reset_index()
    agg = agg.merge(earliest_high, on='trace_name', how='left')
    agg = agg.merge(earliest_low, on='trace_name', how='left')

    # format epochs to Asia/Shanghai localized strings or None
    def fmt_epoch_nullable(x):
        if pd.isna(x) or x is None:
            return None
        return pd.to_datetime(int(x), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')

    agg['earliest_high_ts'] = agg['earliest_high_epoch'].apply(fmt_epoch_nullable)
    agg['earliest_low_ts'] = agg['earliest_low_epoch'].apply(fmt_epoch_nullable)
    agg = agg.drop(columns=['earliest_high_epoch','earliest_low_epoch'])

    # 6) select & sort top 20 by anomalies_total desc (tie-breaker by total_points_in_window)
    top20_traces = agg.sort_values(['anomalies_total','total_points_in_window'], ascending=[False,False]).head(20)

    # 7) select and order requested columns (rename to match exact required names)
    top20_traces = top20_traces.rename(columns={
        'anomalies_high':'anomalies_high',
        'anomalies_low':'anomalies_low'
    })[[
        'trace_name',
        'total_points_in_window',
        'anomalies_high','earliest_high_ts',
        'anomalies_low','earliest_low_ts',
        'anomalies_total',
        'max_value_in_window','min_value_in_window',
        'global_P95','global_P5'
    ]].reset_index(drop=True)

# Display compact top-20 table for checkoutservice-1 trace anomalies
top20_traces

```
Out[11]:
```
# Summarize checkoutservice-1 per-trace anomaly results (compact plain-English string)
try:
    df = top20_traces  # reuse result from previous cell
except NameError:
    summary = "No precomputed results for checkoutservice-1 found in the kernel."
else:
    n = len(df)
    count_by_anom = df['anomalies_total'].value_counts().sort_index(ascending=False).to_dict()
    # compute earliest and latest reported anomaly timestamps among the top rows
    ts_candidates = df['earliest_high_ts'].dropna().tolist() + df['earliest_low_ts'].dropna().tolist()
    earliest = min(ts_candidates) if ts_candidates else None
    latest = max(ts_candidates) if ts_candidates else None
    top3 = df['trace_name'].head(3).tolist()
    # sample numeric examples from first row
    first = df.iloc[0]
    sample_range = f"max={first['max_value_in_window']}, min={first['min_value_in_window']}, P95={first['global_P95']}, P5={first['global_P5']}"
    summary = (
        f"checkoutservice-1 trace anomaly summary (top {n} trace_names):\n"
        f"- Most top trace entries show small but consistent anomalies during the incident window.\n"
        f"- anomalies_total distribution among these {n} traces: {count_by_anom} (e.g. many with 4 anomalies each).\n"
        f"- Earliest anomaly among these traces: {earliest}; latest earliest-anomaly timestamp: {latest}.\n"
        f"- Top 3 affected traces: {top3}.\n"
        f"- Example numeric range for the top trace: {sample_range}.\n"
        f"- Interpretation: checkoutservice-1 exhibits multiple trace metrics with both high- and low-side deviations (typically 2 high + 2 low for many traces), affecting calls to currencyservice, productcatalogservice, shippingservice, and internal/self traces. These anomalies cluster around ~05:30–05:57 CST.\n"
        f"- If you want the detailed top-20 table again, I can display it."
    )

summary

The original code execution output of IPython Kernel is also provided below for reference:

trace_name  total_points_in_window  anomalies_high         earliest_high_ts  anomalies_low          earliest_low_ts  anomalies_total  max_value_in_window  min_value_in_window  global_P95  global_P5
0                    trace.from_root.duration_mean                      28               2  2022-03-21 05:34:00 CST              2  2022-03-21 05:42:00 CST                4             0.012939             0.002657    0.003744   0.003035
1                     trace.from_root.duration_p95                      28               2  2022-03-21 05:32:00 CST              2  2022-03-21 05:42:00 CST                4             0.028252             0.002910    0.004262   0.003319
2                         trace.self.duration_mean                      28               2  2022-03-21 05:32:00 CST              2  2022-03-21 05:45:00 CST                4             0.011392             0.003848    0.009450   0.004002
3                          trace.self.duration_p95                      28               2  2022-03-21 05:50:00 CST              2  2022-03-21 05:43:00 CST                4             0.059271             0.006915    0.034890   0.007096
4         trace.to_currencyservice-2.duration_mean                      27               2  2022-03-21 05:52:00 CST              2  2022-03-21 05:42:00 CST                4             0.000203             0.000098    0.000181   0.000121
5          trace.to_currencyservice-2.duration_p95                      27               2  2022-03-21 05:52:00 CST              2  2022-03-21 05:42:00 CST                4             0.000245             0.000098    0.000190   0.000121
6          trace.to_currencyservice-0.duration_p95                      26               2  2022-03-21 05:41:00 CST              2  2022-03-21 05:31:00 CST                4             0.000724             0.000106    0.000157   0.000119
7          trace.to_currencyservice-1.duration_p95                      25               2  2022-03-21 05:50:00 CST              2  2022-03-21 05:30:00 CST                4             0.000230             0.000111    0.000178   0.000119
8   trace.to_productcatalogservice-2.duration_mean                      24               2  2022-03-21 05:32:00 CST              2  2022-03-21 05:37:00 CST                4             0.005825             0.004308    0.005661   0.004801
9    trace.to_productcatalogservice-2.duration_p95                      24               2  2022-03-21 05:32:00 CST              2  2022-03-21 05:37:00 CST                4             0.005825             0.004587    0.005661   0.004801
10        trace.to_shippingservice-2.duration_mean                      23               2  2022-03-21 05:54:00 CST              2  2022-03-21 05:47:00 CST                4             0.000283             0.000057    0.000099   0.000058
11         trace.to_shippingservice-2.duration_p95                      23               2  2022-03-21 05:54:00 CST              2  2022-03-21 05:47:00 CST                4             0.000283             0.000057    0.000127   0.000058
12  trace.to_productcatalogservice-1.duration_mean                      22               2  2022-03-21 05:57:00 CST              2  2022-03-21 05:50:00 CST                4             0.005925             0.004242    0.005800   0.004588
13   trace.to_productcatalogservice-1.duration_p95                      22               2  2022-03-21 05:57:00 CST              2  2022-03-21 05:30:00 CST                4             0.006638             0.004242    0.005913   0.004749
14        trace.to_currencyservice-0.duration_mean                      26               2  2022-03-21 05:47:00 CST              1  2022-03-21 05:42:00 CST                3             0.000344             0.000106    0.000150   0.000118
15        trace.to_shippingservice-0.duration_mean                      26               2  2022-03-21 05:32:00 CST              1  2022-03-21 05:31:00 CST                3             0.000115             0.000065    0.000090   0.000066
16         trace.to_shippingservice-0.duration_p95                      26               2  2022-03-21 05:32:00 CST              1  2022-03-21 05:31:00 CST                3             0.000115             0.000065    0.000090   0.000066
17        trace.to_currencyservice-1.duration_mean                      25               1  2022-03-21 05:40:00 CST              2  2022-03-21 05:33:00 CST                3             0.000173             0.000111    0.000171   0.000116
18        trace.to_shippingservice-1.duration_mean                      23               2  2022-03-21 05:41:00 CST              1  2022-03-21 05:59:00 CST                3             0.000165             0.000064    0.000093   0.000065
19         trace.to_shippingservice-1.duration_p95                      23               2  2022-03-21 05:41:00 CST              1  2022-03-21 05:59:00 CST                3             0.000254             0.000064    0.000094   0.000065```
```